# HateWiC Foundation Notebook  Supervisor Guide

**24 October checkpoint.** This notebook records the data checks and initial experiments for my Master's research on target-word hateful-sense classification. The main findings and questions are summarised below so the notebook can be reviewed without working through the interactive annotation screens.

| Where to look | What it contains |
| --- | --- |
| Steps 1–5 | Dataset import, majority-vote audit and saved target-position reviews. These interactive review screens have already been completed. |
| Step 6 | Exploratory analysis: class balance, lengths, gloss coverage, annotation counts, duplicate text and frequent content words. |
| Steps 7–9 | Target-position ledger, BERT/HateBERT tokenization and truncation checks, and duplicate-group audit. |
| Step 10 | A **provisional** grouped split for 3,745 target-model examples: 2,622 train, 562 validation and 561 test. One hundred unresolved target positions are listed separately. |
| Step 11 | Three training-only baselines on the common validation IDs. Macro F1: majority 0.344, target-word prior 0.665, and context TF-IDF plus logistic regression 0.693. |
| Step 12 | Synthetic checks for target-subword mean pooling and the planned cosine separation, silhouette and Davies–Bouldin calculations. |

The Step 11 scores are **preliminary validation results**, not final test results. BERT and HateBERT model weights have not yet been trained or compared; the test partition has not been used for model fitting or selection.

**Decisions requested before formal experiments:** confirm how to treat the 100 unresolved target positions, 242 repeated-word cases with rule-selected positions, and 11 majority-labelled cases with a non-standard annotator count; choose linked sense glosses or generated definitions for RO3; then approve the analysis set and split.

**Reviewing and rerunning:** The saved outputs can be read without rerunning the notebook. Please do not use **Run all**, because Steps 4–5 contain completed interactive reviews. The code currently reads from my private Google Drive path (`/content/drive/MyDrive/HateWiC_Project`). Rerunning it from another account requires research-authorised access to the data and a matching path. The dataset and annotation details must not be shared publicly.

---


# HateWiC Step 1: Data Loading and Majority-Vote Verification

Load the source data, aggregate annotations by example, and save the label audit. This step does not train models or create data splits.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, platform, json, hashlib
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import pandas as pd

ROOT = Path('/content/drive/MyDrive/HateWiC_Project')
RAW = ROOT / 'data/raw/HateWiC_IndividualAnnos.csv'
for folder in ['data/raw', 'data/processed', 'results', 'notebooks']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
if not RAW.is_file():
    raise FileNotFoundError('CSV not found. Upload it to MyDrive/HateWiC_Project/data/raw and keep the original filename.')
print('Source file found; size:', RAW.stat().st_size, 'bytes')


In [ ]:
# This CSV is semicolon-delimited. Preserve the string None rather than treating it as missing.
df = pd.read_csv(RAW, sep=';', keep_default_na=False)
required = ['annotation_id', 'annotator_id', 'example_id', 'example', 'term',
            'pos', 'sense_id', 'wiktionary_definition', 'annotation_label',
            'binary_label', 'agree_with_majority_binary']
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
print('Annotation rows:', len(df))
print('Unique examples:', df['example_id'].nunique())
print('Unique target terms:', df['term'].nunique())
print('Column names:', ', '.join(df.columns))
# Do not display source sentences or annotator profiles in shared outputs.


In [ ]:
mapping = {'Not hateful': '0', 'Weakly hateful': '1',
           'Strongly hateful': '1', 'Cannot decide': 'None'}
if not set(df['annotation_label']).issubset(mapping):
    raise ValueError('Unknown source labels found. Check the data before continuing.')
if not df['annotation_label'].map(mapping).eq(df['binary_label'].astype(str)).all():
    raise ValueError('Source labels do not match the binary labels. Check the data.')
if df.duplicated(['example_id', 'annotator_id']).any():
    raise ValueError('Duplicate annotations by the same annotator for an example. Check the data.')
content = ['example', 'term', 'pos', 'sense_id', 'wiktionary_definition']
if df.groupby('example_id')[content].nunique(dropna=False).gt(1).any().any():
    raise ValueError('Text or sense fields differ within an example. Check the data.')

votes = pd.crosstab(df['example_id'], df['binary_label'].astype(str))
votes = votes.reindex(columns=['0', '1', 'None'], fill_value=0)
votes.columns = ['votes_non_hateful', 'votes_hateful', 'votes_cannot_decide']
votes['annotation_count'] = df.groupby('example_id').size()
votes['majority_label'] = pd.Series(pd.NA, index=votes.index, dtype='Int64')
a, b = votes['votes_non_hateful'], votes['votes_hateful']
votes.loc[(a > b) & (a >= 2), 'majority_label'] = 0
votes.loc[(b > a) & (b >= 2), 'majority_label'] = 1
votes['status'] = 'retained'
votes.loc[a.eq(b), 'status'] = 'excluded_tied_binary_votes'
votes.loc[a.ne(b) & a.where(a > b, b).lt(2), 'status'] = 'excluded_fewer_than_two_winning_votes'

lookup = votes['majority_label'].to_dict()
expected = [False if pd.isna(lookup[k]) else str(v) == str(int(lookup[k]))
            for k, v in zip(df['example_id'], df['binary_label'])]
supplied = df['agree_with_majority_binary'].astype(str).str.strip().str.lower()
if not supplied.isin(['true', 'false']).all():
    raise ValueError('Unrecognized majority-flag format. Check the data.')
flag_mismatches = sum(x != y for x, y in zip(expected, supplied.eq('true')))
if flag_mismatches:
    raise ValueError(f'Reconstructed majority flags differ from the source in {flag_mismatches} rows. Check the data before continuing.')

examples = df[['example_id'] + content].drop_duplicates('example_id').set_index('example_id')
master = examples.join(votes).reset_index()
retained = master[master['majority_label'].notna()].copy()
excluded = master[master['majority_label'].isna()].copy()
print('Examples with a majority label:', len(retained))
print('Examples without a majority label:', len(excluded))
print('Class counts (0 = non-hateful, 1 = hateful):', retained['majority_label'].value_counts().sort_index().to_dict())
print('Mismatches with source majority flags:', flag_mismatches)
print('Label audit complete at this stage; target spans, unusual annotator counts, duplicates and data splits are checked in later steps.')


In [ ]:
def new_path(path):
    if not path.exists():
        return path
    n = 2
    while path.with_name(f'{path.stem}_v{n}{path.suffix}').exists():
        n += 1
    return path.with_name(f'{path.stem}_v{n}{path.suffix}')

saved = []
for name, frame in [('majority_candidates.csv', retained),
                    ('excluded_no_majority.csv', excluded),
                    ('annotation_count_review.csv', master[master['annotation_count'].ne(3)])]:
    out = new_path(ROOT / 'data/processed' / name)
    frame.to_csv(out, index=False, encoding='utf-8-sig')
    saved.append(str(out.relative_to(ROOT)))
packages = {}
for package in ['pandas', 'numpy', 'scikit-learn', 'torch', 'transformers']:
    try: packages[package] = version(package)
    except PackageNotFoundError: packages[package] = 'not installed'
audit = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'source_sha256': hashlib.sha256(RAW.read_bytes()).hexdigest(),
    'annotation_rows': len(df), 'unique_examples': int(df['example_id'].nunique()),
    'majority_examples': len(retained), 'excluded_examples': len(excluded),
    'majority_class_counts': {str(k): int(v) for k,v in retained['majority_label'].value_counts().items()},
    'majority_flag_mismatches': flag_mismatches,
    'annotation_count_distribution': {str(k): int(v) for k,v in votes['annotation_count'].value_counts().items()},
    'python': sys.version, 'platform': platform.platform(), 'packages': packages,
    'status': 'Label reconstruction only; not a finalized training dataset or split.'
}
out = new_path(ROOT / 'results/first_run_audit.json')
out.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
saved.append(str(out.relative_to(ROOT)))
print('Saved under HateWiC_Project in Google Drive:')
for name in saved: print(' •', name)
print('Source CSV unchanged. Import and majority-vote checks complete.')


# HateWiC Step 2: Annotator Counts and Target-Position Candidates

Continue from Step 1 by auditing annotator counts and finding candidate character spans for each target. This step checks existing annotations; it does not create hate labels, decide exclusions, train models or rewrite sentences.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re
import pandas as pd

ROOT = Path('/content/drive/MyDrive/HateWiC_Project')
RAW = ROOT / 'data/raw/HateWiC_IndividualAnnos.csv'
MASTER = ROOT / 'data/processed/majority_candidates.csv'
for path in [RAW, MASTER]:
    if not path.is_file():
        raise FileNotFoundError(f'Cannot find {path.name}. Check the Drive path and complete Step 1 first.')
df = pd.read_csv(RAW, sep=';', keep_default_na=False)
master = pd.read_csv(MASTER, keep_default_na=False)
required_raw = {'example_id', 'annotator_id', 'binary_label', 'annotation_label',
                'example', 'term', 'pos', 'sense_id', 'wiktionary_definition'}
required_master = {'example_id', 'example', 'term', 'majority_label'}
if not required_raw.issubset(df) or not required_master.issubset(master):
    raise ValueError('Input columns differ from Step 1. Check the files.')
if master.example_id.duplicated().any() or df.duplicated(['example_id','annotator_id']).any():
    raise ValueError('Duplicate examples or individual annotations found. Check the data.')
# Rebuild and compare labels to detect a mismatched older candidate table.
votes = pd.crosstab(df.example_id, df.binary_label.astype(str)).reindex(columns=['0','1','None'], fill_value=0)
a,b=votes['0'],votes['1']
valid=(a.ne(b)) & ((a>=2)|(b>=2))
expected={int(k): int(votes.loc[k,'1'] > votes.loc[k,'0']) for k in votes.index[valid]}
actual=dict(zip(master.example_id.astype(int),master.majority_label.astype(int)))
if actual != expected:
    raise ValueError('Candidate labels differ from the source CSV majority votes. Rerun Step 1 or select the matching version.')
raw_examples=df.drop_duplicates('example_id').set_index('example_id')
for col in ['example','term']:
    if not master.set_index('example_id')[col].eq(raw_examples[col].reindex(master.example_id).set_axis(master.example_id)).all():
        raise ValueError('Candidate text differs from the source CSV. Check the files.')
print('Input checks passed; majority-label candidates:',len(master))


In [ ]:
sizes=df.groupby('example_id').agg(annotation_count=('annotator_id','size'),
                                   unique_annotators=('annotator_id','nunique'))
count_review=sizes[sizes.annotation_count.ne(3)].join(votes).reset_index()
count_review=count_review.rename(columns={'0':'votes_non_hateful','1':'votes_hateful','None':'votes_cannot_decide'})
count_review['in_majority_candidates']=count_review.example_id.isin(master.example_id)
count_review['review_status']='pending_source_check'
count_review['review_note']=''
print('Annotators per example -> number of examples:')
for n,c in sizes.annotation_count.value_counts().sort_index().items():
    print(f'  {int(n)} -> {int(c)} ')
print('Examples with an annotator count other than 3:',len(count_review))
print('Of these, included among majority-label candidates:',int(count_review.in_majority_candidates.sum()))
print('This audits existing annotations; no new annotators or labels are required.')
print('High annotator counts may indicate shared control items. Check the source; these examples are not automatically removed.')


In [ ]:
def locate_candidates(text, term):
    if not term.strip(): return []
    # Reject target matches that are substrings inside another word.
    # Preserve the text and surface forms; do not automatically select the first of several occurrences.
    pattern=re.escape(term)
    if term[0].isalnum() or term[0]=='_': pattern=r'(?<!\w)' + pattern
    if term[-1].isalnum() or term[-1]=='_': pattern=pattern + r'(?!\w)'
    return [[m.start(),m.end()] for m in re.finditer(pattern,text,flags=re.IGNORECASE)]

alignment=[]
for row in master.itertuples(index=False):
    spans=locate_candidates(str(row.example),str(row.term))
    state=('unique_literal_candidate' if len(spans)==1 else
           'multiple_literal_candidates' if len(spans)>1 else 'no_literal_candidate')
    start,end=spans[0] if len(spans)==1 else (None,None)
    alignment.append({'example_id':row.example_id,'candidate_count':len(spans),
        'candidate_spans':json.dumps(spans),'target_start_candidate':start,
        'target_end_candidate':end,'alignment_status':state})
alignment=pd.DataFrame(alignment)
for col in ['target_start_candidate','target_end_candidate']:
    alignment[col]=alignment[col].astype('Int64')
print('Target-position candidates among majority-label examples:')
for state,n in alignment.alignment_status.value_counts().items(): print(' ',state,':',int(n))
print('unique: one literal candidate, pending spot checks; multiple: choose the intended occurrence; no: check inflection or other surface forms.')
print('These are character-span candidates, not fully verified target spans or BERT subword positions.')


In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=ROOT/'data/processed'/f'review_02_{stamp}'
OUT.mkdir(parents=True,exist_ok=False)
count_review.to_csv(OUT/'annotation_count_review_status.csv',index=False,encoding='utf-8-sig')
alignment.to_csv(OUT/'target_span_candidates.csv',index=False,encoding='utf-8-sig')
# The position-review table omits hate labels and annotator profiles but remains restricted research material.
review_fields=[c for c in ['example_id','example','term','pos','sense_id','wiktionary_definition'] if c in master]
review=master[review_fields].merge(alignment,on='example_id',validate='one_to_one')
review=review[review.alignment_status.ne('unique_literal_candidate')].copy()
review['confirmed_start']='';review['confirmed_end']=''
review['review_decision']='pending';review['review_note']=''
review.to_csv(OUT/'target_span_needs_review.csv',index=False,encoding='utf-8-sig')
unique=master[review_fields].merge(alignment,on='example_id',validate='one_to_one')
unique=unique[unique.alignment_status.eq('unique_literal_candidate')]
unique.sample(n=min(30,len(unique)),random_state=101).to_csv(OUT/'unique_span_spotcheck.csv',index=False,encoding='utf-8-sig')
summary={'source_sha256':hashlib.sha256(RAW.read_bytes()).hexdigest(),
    'candidate_examples':len(master),'non_three_annotation_examples':len(count_review),
    'annotation_count_distribution':{str(k):int(v) for k,v in sizes.annotation_count.value_counts().items()},
    'span_status_counts':{str(k):int(v) for k,v in alignment.alignment_status.value_counts().items()},
    'matching_rule':'case-insensitive literal match with word-edge guards; no normalization or first-match choice',
    'span_convention':'0-based Python character offsets, end exclusive',
    'status':'Candidates only; source checks, manual review and subword alignment remain pending.'}
(OUT/'review_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved to:',str(OUT.relative_to(ROOT)))
print('Source data and Step 1 outputs unchanged. Report the summary counts; no source sentences are needed.')


# HateWiC Step 3: Inflection Matching and Target-Position Candidates

In Step 2, no literal match means that the supplied target form was not found at whole-word boundaries. It does not establish that the example is incorrect: a sentence may contain `cats` for `cat`, or `walked` for `walk`.

This step generates a limited set of regular inflections based on POS and adds candidate character spans. Source sentences and hate labels are unchanged. Matches remain candidates requiring verification.

Run on CPU from top to bottom. Read the Step 1 candidate table and latest Step 2 outputs, verify their hashes and IDs, and save new outputs in a separate `review_03` folder.

- Character offsets are zero-based, with the end position excluded.
- For multiword targets, the rules modify only the final word. Internal and irregular inflections may remain unmatched.
- Display aggregate counts rather than source sentences. Keep the outputs restricted.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import re,json,hashlib
ROOT=Path('/content/drive/MyDrive/HateWiC_Project')
RAW=ROOT/'data/raw/HateWiC_IndividualAnnos.csv'
MASTER=ROOT/'data/processed/majority_candidates.csv'
for p in [RAW,MASTER]:
    if not p.is_file(): raise FileNotFoundError(f'Cannot find {p.name}. Complete Step 1 first.')
folders=sorted(p for p in (ROOT/'data/processed').glob('review_02_*') if (p/'review_summary.json').is_file() and (p/'target_span_candidates.csv').is_file())
if not folders:raise FileNotFoundError('Complete Step 2 and confirm that its outputs are saved in Drive.')
PREV=folders[-1]
summary02=json.loads((PREV/'review_summary.json').read_text())
source_hash=hashlib.sha256(RAW.read_bytes()).hexdigest()
if summary02['source_sha256']!=source_hash:raise ValueError('Step 2 outputs do not match the current source CSV. Check the files.')
df=pd.read_csv(RAW,sep=';',keep_default_na=False)
master=pd.read_csv(MASTER,keep_default_na=False)
old=pd.read_csv(PREV/'target_span_candidates.csv')
if master.example_id.duplicated().any() or old.example_id.duplicated().any() or set(old.example_id)!=set(master.example_id):
    raise ValueError('Step 2 IDs differ from the current candidate table.')
v=pd.crosstab(df.example_id,df.binary_label.astype(str)).reindex(columns=['0','1','None'],fill_value=0)
a,b=v['0'],v['1'];valid=a.ne(b)&((a>=2)|(b>=2))
expected={int(k):int(b.loc[k]>a.loc[k]) for k in v.index[valid]}
if dict(zip(master.example_id.astype(int),master.majority_label.astype(int)))!=expected:
    raise ValueError('Candidate labels differ from the source data.')
raw_examples=df.drop_duplicates('example_id').set_index('example_id')
for row in master.itertuples(index=False):
    for col in ['example','term','pos']:
        if str(getattr(row,col))!=str(raw_examples.loc[row.example_id,col]):
            raise ValueError('Candidate fields differ from the source. Check the files.')
print('Input checks passed. Step 2 output folder:',PREV.name)
print('Majority-label candidates:',len(master))


In [ ]:
def regular_forms(term,pos):
    # Generate candidates only; morphology rules do not establish the intended sense or final position.
    forms={term:'literal'}
    m=re.search(r"[A-Za-z]+$",term)
    if not m:return forms
    base=m.group();prefix=term[:m.start()];w=base.lower();variants=set()
    if pos in ['noun','proper noun','verb']:
        variants.add(base+'s')
        if w.endswith(('s','x','z','ch','sh','o')):variants.add(base+'es')
        if len(w)>1 and w.endswith('y') and w[-2] not in 'aeiou':variants.add(base[:-1]+'ies')
    if pos=='noun' and w.endswith('f'):variants.add(base[:-1]+'ves')
    if pos=='noun' and w.endswith('fe'):variants.add(base[:-2]+'ves')
    if pos=='verb':
        variants.update([base+'ed',base+'ing'])
        if w.endswith('e'):
            variants.add(base+'d')
            if not w.endswith('ee'):variants.add(base[:-1]+'ing')
        if w.endswith('ie'):variants.add(base[:-2]+'ying')
        if len(w)>1 and w.endswith('y') and w[-2] not in 'aeiou':variants.add(base[:-1]+'ied')
        if len(w)>=3 and w[-1] not in 'aeiouwxy' and w[-2] in 'aeiou' and w[-3] not in 'aeiou':
            variants.update([base+base[-1]+'ed',base+base[-1]+'ing'])
    for form in variants:
        if prefix+form!=term:forms[prefix+form]='regular_form_candidate'
    return forms

def spans_for_form(text,form):
    pattern=re.escape(form)
    if form and (form[0].isalnum() or form[0]=='_'):pattern=r'(?<!\w)'+pattern
    if form and (form[-1].isalnum() or form[-1]=='_'):pattern=pattern+r'(?!\w)'
    return [(m.start(),m.end()) for m in re.finditer(pattern,text,re.I)]

rows=[]
for row in master.itertuples(index=False):
    hits={}
    for form,kind in regular_forms(str(row.term),str(row.pos)).items():
        for start,end in spans_for_form(str(row.example),form):
            hits.setdefault((start,end),set()).add(kind)
    spans=sorted(hits)
    literal=[span for span in spans if 'literal' in hits[span]]
    if not spans:state='unresolved_no_candidate'
    elif len(spans)>1:state='multiple_candidates_review'
    elif literal:state='unique_literal_candidate'
    else:state='unique_regular_form_candidate'
    start,end=spans[0] if len(spans)==1 else (None,None)
    rows.append({'example_id':row.example_id,'candidate_count':len(spans),
                 'candidate_spans':json.dumps(spans),'target_start_candidate':start,
                 'target_end_candidate':end,'alignment_status_03':state,
                 'literal_candidate_count':len(literal),'review_status':'pending'})
new=pd.DataFrame(rows)
for c in ['target_start_candidate','target_end_candidate']:new[c]=new[c].astype('Int64')
compare=old[['example_id','alignment_status']].merge(new,on='example_id',validate='one_to_one')
print('Candidate counts after rule-based inflection matching:')
for state,n in new.alignment_status_03.value_counts().items():print(' ',state,':',int(n))
previous_missing=compare.alignment_status.eq('no_literal_candidate')
now_one=compare.alignment_status_03.eq('unique_regular_form_candidate')
print('Previously unmatched examples with one inflection candidate:',int((previous_missing&now_one).sum()))
print('Candidates require verification; the first occurrence is not automatically selected.')


In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=ROOT/'data/processed'/f'review_03_{stamp}'
OUT.mkdir(parents=True,exist_ok=False)
new.to_csv(OUT/'target_span_candidates_v03.csv',index=False,encoding='utf-8-sig')
fields=['example_id','example','term','pos','sense_id','wiktionary_definition']
review=master[fields].merge(compare,on='example_id',validate='one_to_one')
review['confirmed_start']='';review['confirmed_end']='';review['review_note']=''
review[review.alignment_status_03.isin(['multiple_candidates_review','unresolved_no_candidate'])].to_csv(OUT/'remaining_target_review.csv',index=False,encoding='utf-8-sig')
for state,name in [('unique_literal_candidate','literal_spotcheck.csv'),('unique_regular_form_candidate','regular_form_spotcheck.csv')]:
    subset=review[review.alignment_status_03.eq(state)]
    subset.sample(n=min(30,len(subset)),random_state=101).to_csv(OUT/name,index=False,encoding='utf-8-sig')
counts=compare.groupby(['alignment_status','alignment_status_03']).size().reset_index(name='examples')
counts.to_csv(OUT/'matching_status_changes.csv',index=False,encoding='utf-8-sig')
log={'source_sha256':source_hash,'previous_review_folder':PREV.name,'candidate_examples':len(master),
     'status_counts':{str(k):int(v) for k,v in new.alignment_status_03.value_counts().items()},
     'rule_version':'regular-inflection-candidates-v1','position_units':'Python character offsets, zero based, end exclusive',
     'scope':'No original text or labels changed; no examples removed; all positions remain candidates.'}
(OUT/'review_summary.json').write_text(json.dumps(log,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved to:',str(OUT.relative_to(ROOT)))
print('Report the candidate counts before entering decisions or removing examples.')


# HateWiC Step 4: Spot-Check 60 Target Positions

Step 3 produced 3426 single-position candidates, 327 multiple-candidate cases and 92 unmatched cases. This review checks a fixed random sample of 30 literal matches and 30 regular-inflection matches. It does not remove examples or change source sentences or hate labels.

Use CPU. The interface displays real examples, targets and glosses, so keep the notebook restricted. Share aggregate counts rather than example screenshots. After reviewing, Edit → Clear all outputs can remove displayed examples; saved reviews remain in restricted Drive storage.

Check whether the highlight covers the complete target expression corresponding to the supplied term and sense. Inspect inflection, boundaries, accidental substring matches and alternative occurrences. Do not reassess the hate label.

Run this section in order to open the panel. Select Correct, Incorrect or Uncertain for each item, then save. Decisions are written to Drive individually, so an interrupted review can resume. Review 5 items to become familiar with the interface, then finish the sample.


In [ ]:
# Colab usually includes these packages; install missing review-interface components if necessary.
import importlib.util, subprocess, sys
if importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','ipywidgets'])
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json,hashlib,html
ROOT=Path('/content/drive/MyDrive/HateWiC_Project')
RAW=ROOT/'data/raw/HateWiC_IndividualAnnos.csv'
folders=sorted(p for p in (ROOT/'data/processed').glob('review_03_*')
               if (p/'review_summary.json').is_file()
               and (p/'literal_spotcheck.csv').is_file()
               and (p/'regular_form_spotcheck.csv').is_file())
if not folders:raise FileNotFoundError('Complete Step 3 and save both spot-check files first.')
PREV=folders[-1]
source_hash=hashlib.sha256(RAW.read_bytes()).hexdigest()
log=json.loads((PREV/'review_summary.json').read_text())
if log['source_sha256']!=source_hash:raise ValueError('Step 3 does not match the current source CSV. Check the files.')
raw_examples=pd.read_csv(RAW,sep=';',keep_default_na=False).drop_duplicates('example_id').set_index('example_id')

def prepare_queue(literal,regular,raw_examples):
    parts=[]
    for group,frame in [('literal',literal),('regular_form',regular)]:
        frame=frame.copy();frame['audit_group']=group;parts.append(frame)
    queue=pd.concat(parts,ignore_index=True)
    if queue.example_id.duplicated().any():raise ValueError('Duplicate IDs in the spot-check queue.')
    queue=queue.sort_values(['audit_group','example_id']).reset_index(drop=True)
    for row in queue.itertuples(index=False):
        for col in ['example','term','pos','wiktionary_definition']:
            if str(getattr(row,col))!=str(raw_examples.loc[row.example_id,col]):
                raise ValueError('Spot-check text differs from the source CSV.')
        start,end=int(row.target_start_candidate),int(row.target_end_candidate)
        if not 0<=start<end<=len(str(row.example)):raise ValueError('Candidate position is out of bounds.')
    return queue

def render_example(row,index,total):
    text=str(row['example']);start,end=int(row['target_start_candidate']),int(row['target_end_candidate'])
    esc=html.escape
    highlighted=esc(text[:start])+'<mark style="background:#ffe08a;color:#111;padding:2px">'+esc(text[start:end])+'</mark>'+esc(text[end:])
    return (f"<h3>Review {index + 1}/{total} | Example ID {int(row['example_id'])}</h3>"
            f"<p><b>Sample group:</b> {esc(str(row['audit_group']))} <b>Target term:</b> {esc(str(row['term']))} <b>POS:</b> {esc(str(row['pos']))}</p>"
            f"<p><b>Linked gloss:</b> {esc(str(row['wiktionary_definition']))}</p>"
            f'<p style="font-size:18px;line-height:1.8;white-space:pre-wrap">{highlighted}</p>'
            f'<p><b>Candidate span:</b> [{start}, {end}) <b>Highlighted text:</b> {esc(text[start:end])}</p>')

queue=prepare_queue(pd.read_csv(PREV/'literal_spotcheck.csv',keep_default_na=False),
                    pd.read_csv(PREV/'regular_form_spotcheck.csv',keep_default_na=False),raw_examples)
OUT=PREV/'manual_spotcheck_04'
OUT.mkdir(exist_ok=True)
RECORDS=OUT/'review_records.json'
queue_digest=hashlib.sha256(queue[['example_id','audit_group','target_start_candidate','target_end_candidate']].to_json(orient='records').encode()).hexdigest()
state={'source_sha256':source_hash,'queue_sha256':queue_digest,'records':{}}
if RECORDS.is_file():
    state=json.loads(RECORDS.read_text())
    if state.get('source_sha256')!=source_hash or state.get('queue_sha256')!=queue_digest:
        raise ValueError('Saved review records do not match the current queue. Check the files; previous records were not overwritten.')
print('Input checks passed. Spot-check examples:',len(queue))
print('Step 3 source folder:',PREV.name)
print('Saved decisions:',len(state['records']))
print('The next cell opens the review panel.')


# HateWiC Step 5: Restore Saved Reviews and Triage Remaining Cases

Run this section without rerunning Steps 1–4 or interacting with earlier review panels.

The previous loader selected only the latest Step 3 folder, which might not contain completed Step 4 records. This loader checks the Step 3 folders for identical source data, position rules and candidate spans, then restores a matching Step 4 review only if every sampled item has been saved as correct. It does not copy, fabricate or repeat annotations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import importlib.util,subprocess,sys
if importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','ipywidgets'])
import ipywidgets as widgets
from IPython.display import display,HTML,clear_output
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
import json,hashlib,html,re
ROOT=Path('/content/drive/MyDrive/HateWiC_Project')
RAW=ROOT/'data/raw/HateWiC_IndividualAnnos.csv'
from pathlib import Path
import json,hashlib
import pandas as pd


def select_completed_review_run(root,source_hash):
    """Resume a completed review only when its source, rule and full candidates match the latest run."""
    folders=sorted(p for p in (Path(root)/'data/processed').glob('review_03_*')
                   if (p/'review_summary.json').is_file()
                   and (p/'target_span_candidates_v03.csv').is_file())
    if not folders:
        raise FileNotFoundError('Step 3 outputs not found. Check the Drive folder; do not repeat completed spot checks.')

    def signature(folder):
        meta=json.loads((folder/'review_summary.json').read_text())
        if meta.get('source_sha256')!=source_hash:
            raise ValueError('Source CSV hash mismatch')
        frame=pd.read_csv(folder/'target_span_candidates_v03.csv',keep_default_na=False)
        if frame.example_id.duplicated().any():raise ValueError('Duplicate candidate IDs')
        rows=[]
        for row in frame.sort_values('example_id').itertuples(index=False):
            spans=sorted([int(a),int(b)] for a,b in json.loads(row.candidate_spans))
            rows.append([int(row.example_id),str(row.alignment_status_03),spans])
        digest=hashlib.sha256(json.dumps(rows,separators=(',',':')).encode()).hexdigest()
        return (meta.get('rule_version'),digest)

    latest=folders[-1]
    try:
        latest_signature=signature(latest)
    except (OSError,ValueError,KeyError,TypeError,AttributeError) as exc:
        raise FileNotFoundError(f'Cannot verify that the latest Step 3 matches the current source file: {exc}. Previous records were not reused. Check the files.') from exc
    findings=[]
    selected=None
    for folder in reversed(folders):
        try:
            if signature(folder)!=latest_signature:
                findings.append((folder.name,'Candidates or rules differ from the latest Step 3; records not reused'));continue
            record_path=folder/'manual_spotcheck_04/review_records.json'
            if not record_path.is_file():
                findings.append((folder.name,'No Step 4 review records found'));continue
            state=json.loads(record_path.read_text())
            if state.get('source_sha256')!=source_hash:
                findings.append((folder.name,'Step 4 source hash mismatch'));continue
            parts=[]
            for name,group in [('literal_spotcheck.csv','literal'),('regular_form_spotcheck.csv','regular_form')]:
                frame=pd.read_csv(folder/name,keep_default_na=False)
                frame['audit_group']=group;parts.append(frame)
            spot=pd.concat(parts,ignore_index=True).sort_values(['audit_group','example_id']).reset_index(drop=True)
            if spot.empty or spot.example_id.duplicated().any():
                findings.append((folder.name,'Spot-check queue is empty or contains duplicate IDs'));continue
            digest=hashlib.sha256(spot[['example_id','audit_group','target_start_candidate','target_end_candidate']].to_json(orient='records').encode()).hexdigest()
            records=state.get('records',{})
            if state.get('queue_sha256')!=digest:
                findings.append((folder.name,'Step 4 queue hash mismatch'));continue
            expected=set(spot.example_id.astype(int).astype(str))
            if set(records)!=expected:
                findings.append((folder.name,f'Step 4 records incomplete: {len(set(records)&expected)}/{len(expected)}'));continue
            if any(item.get('decision')!='correct' for item in records.values()):
                findings.append((folder.name,'Step 4 contains incorrect or uncertain decisions; review required'));continue
            findings.append((folder.name,f'Step 4 records complete: {len(records)}/{len(expected)}; source and rules match'))
            selected=folder
            break
        except (OSError,ValueError,KeyError,TypeError,AttributeError) as exc:
            findings.append((folder.name,f'Read or verification failed: {type(exc).__name__}'))
    print('Checking the Step 3 folders and saved Step 4 records in Drive:')
    for name,note in findings:print(' ',name,':',note)
    if selected is None:
        raise FileNotFoundError('No completed Step 4 review matches the current data and position rules. Existing records were not overwritten. Check the folder diagnostics above; do not repeat the 60 reviews.')
    print('Restored review folder:',selected.name)
    if selected!=latest:print('No usable review was found for the latest Step 3 folder. Earlier completed records with identical candidates and rules were restored.')
    return selected

if not RAW.is_file():raise FileNotFoundError('Source CSV not found. Check that Drive is mounted with the correct account.')
source_hash=hashlib.sha256(RAW.read_bytes()).hexdigest()
PREV=select_completed_review_run(ROOT,source_hash)
log03=json.loads((PREV/'review_summary.json').read_text())
if log03['source_sha256']!=source_hash:raise ValueError('Source CSV does not match Step 3 outputs.')
p04=PREV/'manual_spotcheck_04/review_records.json'
if not p04.is_file():raise FileNotFoundError('Complete and save the Step 4 review for this Step 3 folder first.')
state04=json.loads(p04.read_text())
if state04.get('source_sha256')!=source_hash:raise ValueError('Step 4 records do not match the current source CSV.')
spotparts=[]
for name,group in [('literal_spotcheck.csv','literal'),('regular_form_spotcheck.csv','regular_form')]:
    part=pd.read_csv(PREV/name,keep_default_na=False);part['audit_group']=group;spotparts.append(part)
spot=pd.concat(spotparts,ignore_index=True).sort_values(['audit_group','example_id']).reset_index(drop=True)
spot_digest=hashlib.sha256(spot[['example_id','audit_group','target_start_candidate','target_end_candidate']].to_json(orient='records').encode()).hexdigest()
if state04.get('queue_sha256')!=spot_digest:raise ValueError('Step 4 spot-check queue does not match Step 3.')
expected_spot=set(spot.example_id.astype(int).astype(str))
if set(state04.get('records',{}))!=expected_spot:raise ValueError('Step 4 contains unsaved items. Complete them before continuing.')
if any(v.get('decision')!='correct' for v in state04['records'].values()):
    raise ValueError('Step 4 contains incorrect or uncertain decisions. Review the rules before proceeding.')
raw=pd.read_csv(RAW,sep=';',keep_default_na=False)
examples=raw.drop_duplicates('example_id').set_index('example_id')
candidates=pd.read_csv(PREV/'target_span_candidates_v03.csv',keep_default_na=False)
v=pd.crosstab(raw.example_id,raw.binary_label.astype(str)).reindex(columns=['0','1','None'],fill_value=0)
a,b=v['0'],v['1'];valid=a.ne(b)&((a>=2)|(b>=2))
if candidates.example_id.duplicated().any() or set(candidates.example_id)!=set(v.index[valid]):
    raise ValueError('Step 3 candidate IDs do not match the current majority-label table.')
# Triage identical repeated surface forms as rule candidates, not manually confirmed positions.
def same_surface_rule_candidates(candidates,examples):
    auto=[];auto_ids=set()
    for row in candidates[candidates.alignment_status_03.eq('multiple_candidates_review')].itertuples(index=False):
        text=str(examples.loc[row.example_id,'example'])
        spans=sorted((int(a),int(b)) for a,b in json.loads(row.candidate_spans))
        if len(spans)<2:raise ValueError('Unexpected number of spans for a multiple-candidate item.')
        for a,b in spans:
            if not 0<=a<b<=len(text):raise ValueError('Span is out of bounds.')
            if text[a].isalnum() and a>0 and (text[a-1].isalnum() or text[a-1]=='_'):
                raise ValueError('Candidate start lies inside a word.')
            if text[b-1].isalnum() and b<len(text) and (text[b].isalnum() or text[b]=='_'):
                raise ValueError('Candidate end lies inside a word.')
        forms={text[a:b].casefold() for a,b in spans}
        if len(forms)!=1:continue
        first,last=spans[0],spans[-1]
        auto_ids.add(int(row.example_id))
        auto.append({'example_id':int(row.example_id),'occurrence_count':len(spans),
            'all_candidate_spans':json.dumps(spans),'target_start_candidate':last[0],
            'target_end_candidate':last[1],'first_start':first[0],'first_end':first[1],
            'alignment_policy':'last_complete_same_surface_candidate_v1',
            'review_status':'rule_assigned_candidate','position_is_gold':False,
            'ambiguity_flag':'multiple_occurrences_not_disambiguated'})
    return pd.DataFrame(auto),auto_ids

rule_candidates,rule_ids=same_surface_rule_candidates(candidates,examples)
remaining=candidates[candidates.alignment_status_03.isin(['multiple_candidates_review','unresolved_no_candidate']) & ~candidates.example_id.isin(rule_ids)].copy()
queue=remaining.merge(examples.reset_index()[['example_id','example','term','pos','sense_id','wiktionary_definition']],on='example_id',validate='one_to_one')
queue=queue.sort_values(['alignment_status_03','example_id']).reset_index(drop=True)
OUT=PREV/'target_review_reduced_05b';OUT.mkdir(exist_ok=True)
RECORDS=OUT/'target_review_records.json'
queue_digest=hashlib.sha256(queue[['example_id','alignment_status_03','candidate_spans']].to_json(orient='records').encode()).hexdigest()
state={'source_sha256':source_hash,'queue_sha256':queue_digest,'records':{}}
if RECORDS.is_file():
    state=json.loads(RECORDS.read_text())
    if state.get('source_sha256')!=source_hash or state.get('queue_sha256')!=queue_digest:
        raise ValueError('Saved reviews do not match the current queue. Previous records were not overwritten.')
print('Step 4 spot checks complete; all sampled positions marked correct. This finding applies to the sample.')
print('Repeated-surface candidates assigned by a rule (not gold spans):',len(rule_candidates))
print('Examples in the manual review queue:',len(queue))
print('Examples with multiple candidate positions:',int(queue.alignment_status_03.eq('multiple_candidates_review').sum()))
print('Examples without a rule-based candidate:',int(queue.alignment_status_03.eq('unresolved_no_candidate').sum()))
print('Saved decisions in this review:',len(state['records']))

# Save rule candidates, first/last positions and policy separately; do not overwrite Step 3 or earlier Step 5 files.
rule_candidates.to_csv(OUT/'rule_assigned_repeated_candidates.csv',index=False,encoding='utf-8-sig')
rule_candidates[['example_id','first_start','first_end','target_start_candidate','target_end_candidate']].to_csv(OUT/'first_last_sensitivity_positions.csv',index=False,encoding='utf-8-sig')
triage={'source_sha256':source_hash,'previous_review_folder':PREV.name,
        'rule_assigned_candidates':len(rule_candidates),'manual_review_queue':len(queue),
        'rule_version':'last_complete_same_surface_candidate_v1','rule_position_is_gold':False,
        'scope':'Proposed preprocessing rule only; no final training input is built. Original text and labels unchanged.',
        'source_reference':'https://github.com/SanneHoeken/HateWiC/blob/main/code/embeddings.py',
        'source_difference':'Inspired by last matching indices in original code; uses complete surface forms without editing context, not an exact reproduction.'}
(OUT/'triage_policy.json').write_text(json.dumps(triage,ensure_ascii=False,indent=2),encoding='utf-8')
print('Triage results saved to:',str(OUT.relative_to(ROOT)))
print('The next review panel contains the remaining queue. Review the triage counts before continuing.')


In [ ]:
# Treat U+02BC quotes and U+01C3 exclamation marks as boundaries, without changing the source sentence.
def _target_word_char(char):
    return char not in ('ʼ','ǃ') and (char.isalnum() or char == '_')

def validate_span(text,start,end):
    if not isinstance(start,int) or not isinstance(end,int) or not 0<=start<end<=len(text):
        raise ValueError('Invalid position format or out-of-bounds span.')
    if text[start].isalnum() and start>0 and _target_word_char(text[start-1]):
        raise ValueError('The span starts inside a word. Select the complete surface expression.')
    if text[end-1].isalnum() and end<len(text) and _target_word_char(text[end]):
        raise ValueError('The span ends inside a word. Select the complete surface expression.')
    return (start,end)

def manual_occurrences(text,surface):
    surface=surface.strip()
    if not surface:return []
    found=[]
    for m in re.finditer(re.escape(surface),text):
        try:found.append(validate_span(text,m.start(),m.end()))
        except ValueError:pass
    return found

def highlighted_text(text,span=None):
    if span is None:return html.escape(text)
    start,end=validate_span(text,*span)
    return html.escape(text[:start])+'<mark style="background:#ffe08a;color:#111">'+html.escape(text[start:end])+'</mark>'+html.escape(text[end:])

def original_candidates(row):
    spans=json.loads(row['candidate_spans'])
    return [validate_span(str(row['example']),int(a),int(b)) for a,b in spans]

assert validate_span('a ‘bumʼ”',3,6)==(3,6)
assert manual_occurrences('gad\u01c3','gad')==[(0,3)]
assert manual_occurrences('Two cats sleep.','cat')==[]
assert manual_occurrences('Two cats sleep.','cats')==[(4,8)]
for row in queue.to_dict('records'):original_candidates(row)
for key,item in state['records'].items():
    if item.get('decision')=='confirmed':
        r=queue[queue.example_id.astype(str).eq(key)]
        if len(r)!=1:raise ValueError('A saved review ID is not in the current queue.')
        text=str(r.iloc[0]['example']);span=validate_span(text,int(item['start']),int(item['end']))
        if text[span[0]:span[1]]!=item.get('surface'):raise ValueError('Previously confirmed text does not match the source span.')
print('Span checks passed, including U+02BC quote and U+01C3 exclamation boundaries and whole-word checks. Source sentences and labels unchanged.')


In [ ]:
if len(queue)==0:raise ValueError('No items in the review queue.')
position=next((i for i,r in queue.iterrows() if str(int(r.example_id)) not in state['records']),len(queue)-1)
panel=widgets.Output();preview=widgets.HTML();status=widgets.HTML();hint=widgets.HTML()
span_choice=widgets.Dropdown(options=[('Select a target position','')],value='',description='Position:',layout=widgets.Layout(width='95%'))
surface_search=widgets.Text(value='',description='Surface text:',placeholder='If no candidate fits, copy the complete expression from the sentence, preserving case',layout=widgets.Layout(width='95%'))
search_button=widgets.Button(description='Find surface text')
decision=widgets.Dropdown(options=[('Select a decision',''),('Target position confirmed','confirmed'),('Uncertain','uncertain'),('No identifiable target expression','no_identifiable_target')],value='',description='Decision:',layout=widgets.Layout(width='430px'))
note=widgets.Textarea(value='',description='Note:',placeholder='Explain uncertain or unlocatable cases; for manual spans, note the inflection or spelling change.',layout=widgets.Layout(width='95%',height='70px'))
back=widgets.Button(description='Previous');forward=widgets.Button(description='Next (do not save)')
save=widgets.Button(description='Save and next',button_style='success')
refresh=widgets.Button(description='Refresh counts')
active_sources={}

def records_frame():
    rows=[]
    for r in queue.to_dict('records'):
        item=state['records'].get(str(int(r['example_id'])),{})
        rows.append({'example_id':int(r['example_id']),'previous_status':r['alignment_status_03'],
            'decision':item.get('decision','pending'),'confirmed_start':item.get('start',''),
            'confirmed_end':item.get('end',''),'confirmed_surface':item.get('surface',''),
            'selection_source':item.get('selection_source',''),'note':item.get('note',''),
            'reviewed_at_utc':item.get('reviewed_at_utc','')})
    return pd.DataFrame(rows)

def update_progress(_=None):
    frame=records_frame();counts=frame.decision.value_counts().to_dict()
    status.value=(f"<b>Reviewed:</b> {int(frame.decision.ne('pending').sum())}/{len(frame)}"
                  f" | Confirmed {counts.get('confirmed', 0)} | Uncertain {counts.get('uncertain', 0)}"
                  f" | Unlocated {counts.get('no_identifiable_target', 0)} | Pending {counts.get('pending', 0)}"
                  '<br>Reviewed items may remain unresolved. Uncertain and unlocated items are retained.')

def persist():
    pending=OUT/'target_review_records.pending.json'
    pending.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
    if RECORDS.exists():(OUT/'target_review_records.previous.json').write_bytes(RECORDS.read_bytes())
    pending.replace(RECORDS)
    frame=records_frame()
    frame.to_csv(OUT/'target_review_decisions.csv',index=False,encoding='utf-8-sig')
    frame.groupby(['previous_status','decision']).size().reset_index(name='examples').to_csv(OUT/'target_review_progress.csv',index=False,encoding='utf-8-sig')

def update_preview(change=None):
    value=span_choice.value
    if not value:preview.value='';return
    start,end=map(int,value.split(':'));text=str(queue.iloc[position]['example'])
    preview.value='<p style="font-size:18px;line-height:1.8;white-space:pre-wrap">'+highlighted_text(text,(start,end))+'</p>'

span_choice.observe(update_preview,names='value')

def show():
    global active_sources
    row=queue.iloc[position];text=str(row['example']);old=state['records'].get(str(int(row['example_id'])),{})
    options=[('Select a target position','')];active_sources={}
    for i,(start,end) in enumerate(original_candidates(row)):
        key=f'{start}:{end}';options.append((f'Candidate {i + 1} [{start}, {end})  '+text[start:end],key));active_sources[key]='rule_candidate'
    if old.get('decision')=='confirmed':
        key=f'{old["start"]}:{old["end"]}'
        if key not in active_sources:options.append((f"Previously confirmed [{old['start']}, {old['end']}) "+old['surface'],key))
        active_sources[key]=old.get('selection_source','saved_manual')
    span_choice.options=options
    span_choice.value=f'{old["start"]}:{old["end"]}' if old.get('decision')=='confirmed' else ''
    decision.value=old.get('decision','');note.value=old.get('note','');surface_search.value='';hint.value=''
    with panel:
        clear_output(wait=True)
        display(HTML(f"<h3>Review {position + 1}/{len(queue)} | Example ID {int(row['example_id'])}</h3>"
          f"<p><b>Type:</b> {html.escape(str(row['alignment_status_03']))} <b>Target term:</b> {html.escape(str(row['term']))} <b>POS:</b> {html.escape(str(row['pos']))}</p>"
          f"<p><b>Linked gloss:</b> {html.escape(str(row['wiktionary_definition']))}</p>"
          f'<p style="font-size:18px;line-height:1.8;white-space:pre-wrap">{html.escape(text)}</p>'))
    update_preview();update_progress()

def search_surface(_):
    text=str(queue.iloc[position]['example'])
    hits=manual_occurrences(text,surface_search.value)
    if not hits:
        hint.value='No whole-word match found. Check the copied text; select Uncertain if the position cannot be established.';return
    options=list(span_choice.options);known={value for label,value in options}
    for i,(start,end) in enumerate(hits):
        key=f'{start}:{end}'
        if key not in known:
            options.append((f'Surface match {i + 1} [{start}, {end}) '+text[start:end],key));known.add(key);active_sources[key]='manual_surface_search'
    span_choice.options=options
    span_choice.value=f'{hits[0][0]}:{hits[0][1]}' if len(hits)==1 else ''
    hint.value=f'Found {len(hits)} matches. Check the highlighted preview; do not default to the first of several identical occurrences.'

def move(delta):
    global position
    position=min(max(position+delta,0),len(queue)-1);show()

def save_next(_):
    global position
    if not decision.value:hint.value='Not saved. Select a decision first.';return
    text=str(queue.iloc[position]['example']);start=end=None;surface='';origin=''
    if decision.value=='confirmed':
        if not span_choice.value:hint.value='Not saved. Select and check the target position first.';return
        start,end=map(int,span_choice.value.split(':'));validate_span(text,start,end)
        surface=text[start:end];origin=active_sources.get(span_choice.value,'unknown')
        if origin=='manual_surface_search' and not note.value.strip():
            hint.value='Not saved. Add a brief reason for the manually located span.';return
    elif not note.value.strip():
        hint.value='Not saved. Explain the uncertain or unlocated decision.';return
    key=str(int(queue.iloc[position]['example_id']))
    state['records'][key]={'decision':decision.value,'start':start,'end':end,'surface':surface,
        'selection_source':origin,'note':note.value.strip(),'reviewed_at_utc':datetime.now(timezone.utc).isoformat()}
    persist()
    pending=[i for i in range(position+1,len(queue)) if str(int(queue.iloc[i].example_id)) not in state['records']]
    if not pending:pending=[i for i in range(len(queue)) if str(int(queue.iloc[i].example_id)) not in state['records']]
    if pending:position=pending[0]
    show()
    if not pending:status.value += '<br><b>All decisions recorded.</b> Review the counts; uncertain and unlocated items still need follow-up.'

search_button.on_click(search_surface)
back.on_click(lambda _:move(-1));forward.on_click(lambda _:move(1))
save.on_click(save_next);refresh.on_click(update_progress)
display(widgets.VBox([panel,span_choice,preview,surface_search,search_button,hint,decision,note,
                     widgets.HBox([back,forward,save]),refresh,status]))
show()


In [ ]:
# Step 5 verification reads saved records without overwriting reviews, relabeling or removing examples.
from pathlib import Path
import json, hashlib
import pandas as pd
saved_review = json.loads(RECORDS.read_text(encoding='utf-8'))
saved_frame = pd.read_csv(OUT/'target_review_decisions.csv', keep_default_na=False)
expected_ids = set(queue.example_id.astype(int).astype(str))
assert set(saved_review['records']) == expected_ids, 'Saved JSON IDs do not cover the complete queue'
assert len(saved_frame) == len(queue) and not saved_frame.example_id.duplicated().any(), 'Unexpected CSV row count or IDs'
assert set(saved_frame.example_id.astype(int).astype(str)) == expected_ids
assert saved_frame.decision.ne('pending').all(), 'Pending review items remain'
assert hashlib.sha256(RAW.read_bytes()).hexdigest() == saved_review['source_sha256'], 'Source CSV hash mismatch'
queue_lookup=queue.set_index('example_id')
for example_key, record in saved_review['records'].items():
    csv_row=saved_frame.loc[saved_frame.example_id.astype(str).eq(example_key)].iloc[0]
    assert csv_row.decision == record['decision'], 'JSON and CSV decisions differ'
    if record['decision']=='confirmed':
        text=str(queue_lookup.loc[int(example_key),'example'])
        start,end=int(record['start']),int(record['end'])
        assert 0 <= start < end <= len(text)
        assert text[start:end] == record['surface'], 'Confirmed span does not match the source text'
        assert float(csv_row.confirmed_start)==start and float(csv_row.confirmed_end)==end
counts=saved_frame.decision.value_counts()
print('Step 5: saved results verified')
print(f'Recorded:{len(saved_frame)}/{len(queue)}; pending:{int(saved_frame.decision.eq("pending").sum())}')
print(f'Confirmed positions:{counts.get("confirmed",0)}; uncertain:{counts.get("uncertain",0)}; no identifiable target:{counts.get("no_identifiable_target",0)}')
print('JSON and CSV decisions agree; confirmed spans match the source text; source CSV hash verified.')
print('Recorded reviews may still contain unresolved positions. Check those records and conflicting notes before merging spans, auditing duplicates and starting EDA.')
print('Review includes AI assistance and does not establish independent human annotation or agreement between two reviewers.')
print('Records folder:',str(OUT.relative_to(ROOT)))


# HateWiC Step 6: Exploratory Data Analysis

Prepare reproducible summary tables and figures for supervisor review. CPU is sufficient.

This section is part of the unified notebook. It uses independent variables, reads saved inputs and does not overwrite review records. Run one cell at a time; do not rerun Steps 1–5.

Analyse all source examples and the reconstructed majority-label candidates, retaining unresolved target positions. This section does not build final model inputs, create train/validation/test splits or train models.

Outputs are saved under `HateWiC_Project/results/eda_06_<timestamp>/` in Drive. Source and review files are read-only.

Keep the dataset, frequency tables and notebook restricted. Display aggregate counts and figures without source sentences or annotator profiles. Word frequencies are descriptive, not rules for assigning hate labels.

## 1. Connect Google Drive

Run the next cell. A successful connection displays `Mounted at` or `Drive already mounted`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Read and Verify Inputs

Reconstruct majority votes and compare them with the source flags. Read the fixed Step 5 review folder rather than a newly created empty folder. The reference status is 77 confirmed, 91 uncertain and 9 unlocated; later review updates should display their actual counts.

Package versions are recorded. This section does not install models or download tokenizers. If an analysis package is missing, resolve it before continuing.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, json, platform, re, sys, unicodedata
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

e06_root = Path('/content/drive/MyDrive/HateWiC_Project')
e06_raw_path = e06_root/'data/raw/HateWiC_IndividualAnnos.csv'
e06_review_dir = e06_root/'data/processed/review_03_20261019T022652709994Z/target_review_reduced_05b'
e06_record_path = e06_review_dir/'target_review_records.json'
e06_csv_path = e06_review_dir/'target_review_decisions.csv'
for e06_p in [e06_raw_path, e06_record_path, e06_csv_path]:
    if not e06_p.is_file():
        raise FileNotFoundError(f'Missing input file: {e06_p.name}. Check the Drive account and path; do not repeat the reviews.')
e06_input_paths = [e06_raw_path, e06_record_path, e06_csv_path]
e06_input_hashes = {str(p.relative_to(e06_root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in e06_input_paths}
e06_review = json.loads(e06_record_path.read_text(encoding='utf-8'))
if e06_review['source_sha256'] != e06_input_hashes[str(e06_raw_path.relative_to(e06_root))]:
    raise ValueError('Source file hash differs from the Step 5 review. Analysis stopped; inputs were not overwritten.')
e06_fields = ['annotation_id','annotator_id','example_id','example','term','pos','sense_id',
              'wiktionary_definition','annotation_label','binary_label','agree_with_majority_binary']
e06_raw = pd.read_csv(e06_raw_path, sep=';', keep_default_na=False, usecols=e06_fields)
e06_raw['example_id'] = e06_raw.example_id.astype(str)
e06_raw['binary_label'] = e06_raw.binary_label.astype(str)
if e06_raw.duplicated(['example_id','annotator_id']).any():
    raise ValueError('Duplicate annotations by the same annotator for an example. Check the data first.')
e06_content = ['example','term','pos','sense_id','wiktionary_definition']
if e06_raw.groupby('example_id')[e06_content].nunique(dropna=False).gt(1).any().any():
    raise ValueError('Source text or sense fields differ within an example. Check the data first.')
e06_label_map = {'Not hateful':'0','Weakly hateful':'1','Strongly hateful':'1','Cannot decide':'None'}
if not e06_raw.annotation_label.isin(e06_label_map).all():
    raise ValueError('Unknown labels found.')
if not e06_raw.annotation_label.map(e06_label_map).eq(e06_raw.binary_label).all():
    raise ValueError('Source labels do not match the binary mapping.')
e06_votes = pd.crosstab(e06_raw.example_id, e06_raw.binary_label).reindex(columns=['0','1','None'], fill_value=0)
e06_votes.columns = ['votes_non_hateful','votes_hateful','votes_cannot_decide']
e06_votes['annotation_count'] = e06_raw.groupby('example_id').size()
e06_a, e06_b = e06_votes.votes_non_hateful, e06_votes.votes_hateful
e06_valid = e06_a.ne(e06_b) & ((e06_a >= 2) | (e06_b >= 2))
e06_votes['majority_label'] = pd.Series(pd.NA, index=e06_votes.index, dtype='Int64')
e06_votes.loc[e06_valid,'majority_label'] = (e06_b[e06_valid] > e06_a[e06_valid]).astype(int)
e06_vote_lookup = e06_votes.majority_label.to_dict()
e06_expected_flags = pd.Series([
    False if pd.isna(e06_vote_lookup[k]) else str(v) == str(int(e06_vote_lookup[k]))
    for k,v in zip(e06_raw.example_id, e06_raw.binary_label)], index=e06_raw.index)
e06_supplied_flags = e06_raw.agree_with_majority_binary.astype(str).str.strip().str.lower()
if not e06_supplied_flags.isin(['true','false']).all() or not e06_supplied_flags.eq('true').eq(e06_expected_flags).all():
    raise ValueError('Reconstructed majority votes differ from the source flags.')
e06_all = e06_raw[['example_id']+e06_content].drop_duplicates('example_id').set_index('example_id').join(e06_votes).reset_index()
e06_master = e06_all[e06_all.majority_label.notna()].copy().reset_index(drop=True)
e06_master['majority_label'] = e06_master.majority_label.astype(int)
e06_decisions = pd.read_csv(e06_csv_path, keep_default_na=False, dtype={'example_id': str})
if e06_decisions.example_id.duplicated().any() or set(e06_decisions.example_id) != set(e06_review['records']):
    raise ValueError('Review JSON and CSV example IDs differ.')
if not set(e06_decisions.example_id).issubset(set(e06_master.example_id)):
    raise ValueError('Review IDs include examples outside the majority-label table.')
e06_lookup = e06_master.set_index('example_id')
for e06_r in e06_decisions.itertuples(index=False):
    e06_item = e06_review['records'][e06_r.example_id]
    if e06_item['decision'] != e06_r.decision or e06_item['note'] != e06_r.note:
        raise ValueError('Review JSON and CSV decisions or notes differ.')
    if e06_item['decision'] == 'confirmed':
        e06_s, e06_e = e06_item['start'], e06_item['end']
        e06_text = str(e06_lookup.loc[e06_r.example_id,'example'])
        if not (isinstance(e06_s,int) and isinstance(e06_e,int) and 0 <= e06_s < e06_e <= len(e06_text)):
            raise ValueError('Confirmed position is out of bounds.')
        if e06_text[e06_s:e06_e] != e06_item['surface'] or e06_r.confirmed_surface != e06_item['surface']:
            raise ValueError('Confirmed position does not match the source sentence.')
        if float(e06_r.confirmed_start) != e06_s or float(e06_r.confirmed_end) != e06_e:
            raise ValueError('Character spans differ between JSON and CSV.')
e06_review_counts = {str(k): int(v) for k,v in e06_decisions.decision.value_counts().items()}
e06_versions = {p: version(p) for p in ['pandas','numpy','matplotlib']}
print('Step 6: input checks passed')
print('Source annotation rows:',len(e06_raw),'; unique examples:',len(e06_all))
print('Majority-label candidates:',len(e06_master),'; examples without a majority label:',len(e06_all)-len(e06_master))
print('Review status:',e06_review_counts)
print('Environment versions:',e06_versions)
print('At this stage, models have not been trained and final model inputs and test partitions have not been created.')

## 3. Lengths, Labels, Annotator Counts and Sense Support

Report whitespace-separated item counts, regex word counts and character counts. These are not BERT subword counts; model token lengths and truncation rates are checked later.

Retain and report examples with an annotator count other than three separately. Vote support is the majority proportion among decidable binary votes, not Fleiss kappa or model performance.

Counts of both labels for the same target support the RO1 feasibility check. Mixed term labels alone do not establish a hate-heterogeneous subgroup or define training-unseen terms.


In [ ]:
def e06_words(text):
    return re.findall(r"[^\W_]+(?:['’][^\W_]+)*", str(text), flags=re.UNICODE)

def e06_norm(text):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC',str(text)).casefold()).strip()

e06_metrics = e06_master.copy()
for e06_col, e06_prefix in [('example','context'),('wiktionary_definition','gloss'),('term','target')]:
    e06_metrics[e06_prefix+'_characters'] = e06_metrics[e06_col].astype(str).map(len)
    e06_metrics[e06_prefix+'_whitespace_words'] = e06_metrics[e06_col].astype(str).map(lambda t: len(t.split()))
    e06_metrics[e06_prefix+'_lexical_words'] = e06_metrics[e06_col].map(lambda t: len(e06_words(t)))
e06_metrics['context_mean_word_characters'] = e06_metrics.example.map(lambda t: float(np.mean([len(w) for w in e06_words(t)])) if e06_words(t) else np.nan)
e06_metrics['has_gloss'] = e06_metrics.wiktionary_definition.astype(str).str.strip().ne('')
e06_metrics['term_key'] = e06_metrics.term.map(e06_norm)
e06_metrics['pos_key'] = e06_metrics.pos.map(e06_norm)
e06_metrics['has_sense_id'] = e06_metrics.sense_id.astype(str).str.strip().ne('')
e06_class = e06_metrics.groupby('majority_label').size().rename('examples').reset_index()
e06_class['fraction'] = e06_class.examples/len(e06_metrics)
e06_length_cols = [c for c in e06_metrics if c.endswith(('_characters','_whitespace_words','_lexical_words'))]
e06_length_summary = e06_metrics[e06_length_cols].describe(percentiles=[.05,.25,.5,.75,.95,.99]).T.rename_axis('measure').reset_index()
e06_by_class = e06_metrics.groupby('majority_label')[e06_length_cols].agg(['mean','median','min','max'])
e06_by_class.columns = [f'{a}_{b}' for a,b in e06_by_class.columns]
e06_by_class = e06_by_class.reset_index()
e06_annotation_counts = e06_all.annotation_count.value_counts().sort_index().rename_axis('annotation_count').reset_index(name='examples')
e06_annotation_audit = e06_all[['example_id','annotation_count','votes_non_hateful','votes_hateful','votes_cannot_decide','majority_label']].copy()
e06_annotation_audit['non_three_annotations'] = e06_annotation_audit.annotation_count.ne(3)
e06_annotation_audit['valid_binary_votes'] = e06_annotation_audit.votes_non_hateful+e06_annotation_audit.votes_hateful
e06_annotation_audit['winning_binary_vote_fraction'] = e06_annotation_audit[['votes_non_hateful','votes_hateful']].max(axis=1)/e06_annotation_audit.valid_binary_votes.replace(0,np.nan)
e06_annotation_audit['all_annotators_same_binary_label'] = ((e06_annotation_audit.votes_non_hateful==e06_annotation_audit.annotation_count) | (e06_annotation_audit.votes_hateful==e06_annotation_audit.annotation_count))
e06_pos = e06_metrics.groupby(['pos','majority_label']).size().reset_index(name='examples')
e06_term_support = pd.crosstab([e06_metrics.term_key,e06_metrics.pos_key],e06_metrics.majority_label).reindex(columns=[0,1],fill_value=0)
e06_term_support.columns = ['non_hateful_examples','hateful_examples']
e06_term_support['examples'] = e06_term_support.sum(axis=1)
e06_term_support['both_labels_observed'] = e06_term_support.non_hateful_examples.gt(0)&e06_term_support.hateful_examples.gt(0)
e06_term_support['at_least_two_per_class'] = e06_term_support.non_hateful_examples.ge(2)&e06_term_support.hateful_examples.ge(2)
e06_term_support['at_least_five_per_class'] = e06_term_support.non_hateful_examples.ge(5)&e06_term_support.hateful_examples.ge(5)
e06_term_support = e06_term_support.reset_index()
e06_sense_support = e06_metrics.groupby(['term_key','pos_key','sense_id']).majority_label.agg(examples='size',observed_labels='nunique').reset_index()
e06_sense_support['sense_label_mixed'] = e06_sense_support.observed_labels.gt(1)
e06_coverage = pd.DataFrame([
    {'population':'all_original_examples','examples':len(e06_all),'with_nonempty_gloss':int(e06_all.wiktionary_definition.astype(str).str.strip().ne('').sum())},
    {'population':'majority_candidates','examples':len(e06_metrics),'with_nonempty_gloss':int(e06_metrics.has_gloss.sum())}])
e06_coverage['gloss_fraction'] = e06_coverage.with_nonempty_gloss/e06_coverage.examples
print('Class distribution (0 = non-hateful, 1 = hateful):')
print(e06_class.to_string(index=False))
print('Unique source target terms among majority-label candidates:',e06_metrics.term.nunique())
print('Normalized term and POS groups:',len(e06_term_support))
print('Term and POS groups with both labels:',int(e06_term_support.both_labels_observed.sum()))
print('Term and POS groups with at least 2 examples per class:',int(e06_term_support.at_least_two_per_class.sum()))
print('Examples with an annotator count other than 3:',int(e06_annotation_audit.non_three_annotations.sum()),'(exclusion not yet decided)')
print('Gloss coverage:')
print(e06_coverage.to_string(index=False))
print('Training-unseen terms are not yet defined at this step; a fixed train/validation/test split is required.')

## 4. Duplicate Text and Content-Word Frequencies

Report exact source duplicates, duplicates after NFKC/case/whitespace normalization, and candidate duplicates after ignoring punctuation. These checks preserve the source text and do not automatically remove examples. They do not identify every near duplicate; source relationships and additional similarity checks are needed before splitting.

Use fixed English tokenization, a fixed stopword list and words containing at least 2 letters. Frequency tables can support word clouds; individual terms are not displayed by default. Save target-term frequencies separately from whole-context content-word frequencies.


In [ ]:
e06_work = e06_metrics.copy()
e06_work['exact_key'] = e06_work.example.astype(str)
e06_work['normalized_key'] = e06_work.example.map(e06_norm)
e06_work['punctuation_insensitive_key'] = e06_work.normalized_key.map(lambda t: re.sub(r'\W+', ' ', t, flags=re.UNICODE).strip())
e06_duplicate_rows = []
e06_duplicate_summaries = []
for e06_kind in ['exact_key','normalized_key','punctuation_insensitive_key']:
    e06_group_sizes = e06_work.groupby(e06_kind).size()
    e06_duplicate_keys = set(e06_group_sizes[e06_group_sizes.gt(1)].index)
    e06_duplicate_summaries.append({'rule':e06_kind,'duplicate_groups':len(e06_duplicate_keys),
        'examples_in_groups':int(e06_group_sizes[e06_group_sizes.gt(1)].sum()),
        'extra_rows_beyond_one_per_group':int((e06_group_sizes[e06_group_sizes.gt(1)]-1).sum())})
    for e06_key,e06_group in e06_work[e06_work[e06_kind].isin(e06_duplicate_keys)].groupby(e06_kind):
        e06_gid = hashlib.sha256((e06_kind+'\0'+e06_key).encode('utf-8')).hexdigest()[:20]
        for e06_row in e06_group.itertuples(index=False):
            e06_duplicate_rows.append({'rule':e06_kind,'group_id':e06_gid,'example_id':e06_row.example_id,
                'group_size':len(e06_group),'distinct_targets':e06_group.term_key.nunique(),
                'distinct_labels':e06_group.majority_label.nunique(),
                'same_target_and_pos_mixed_labels':bool(e06_group.groupby(['term_key','pos_key']).majority_label.nunique().gt(1).any())})
e06_duplicates = pd.DataFrame(e06_duplicate_rows,columns=['rule','group_id','example_id','group_size','distinct_targets','distinct_labels','same_target_and_pos_mixed_labels'])
e06_duplicate_summary = pd.DataFrame(e06_duplicate_summaries)
e06_stopwords = set('a an the and or but if while of to in on at for from with by as is are was were be been being it its this that these those i me my we us our you your he him his she her they them their who whom which what where when why how not no nor so than then too very can could may might must shall should will would do does did have has had into out up down over under again further here there once only also just all any both each few more most other some such own same s t d ll m re ve y'.split())
def e06_content_tokens(text):
    return [w for w in re.findall(r"[a-z]+(?:['’][a-z]+)*", str(text).casefold())
            if len(w)>=2 and w not in e06_stopwords]
e06_frequencies = []
for e06_population,e06_frame in [('all_majority',e06_metrics),('label_0',e06_metrics[e06_metrics.majority_label.eq(0)]),('label_1',e06_metrics[e06_metrics.majority_label.eq(1)])]:
    e06_tf,e06_df = Counter(),Counter()
    for e06_text in e06_frame.example:
        e06_tokens=e06_content_tokens(e06_text);e06_tf.update(e06_tokens);e06_df.update(set(e06_tokens))
    for e06_token,e06_count in sorted(e06_tf.items(),key=lambda x:(-x[1],x[0])):
        e06_frequencies.append({'population':e06_population,'token':e06_token,'count':e06_count,
            'document_frequency':e06_df[e06_token],'documents':len(e06_frame)})
e06_content_frequency = pd.DataFrame(e06_frequencies)
print('Duplicate-text candidates (no automatic removal):')
print(e06_duplicate_summary.to_string(index=False))
print('Content-word frequencies, frequencies by label and target-term frequencies prepared. Frequency does not measure hate intensity or establish causation.')

## 5. Draw Summary Figures

Plot class counts, context lengths, gloss lengths, average word lengths and annotator counts. Figures show aggregate data with English labels. Length histograms retain the full range, including long texts.


In [ ]:
e06_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e06_out = e06_root/'results'/f'eda_06_{e06_stamp}'
e06_out.mkdir(parents=True,exist_ok=False)
(e06_out/'figures').mkdir()
plt.style.use('default')
e06_figure_paths=[]
def e06_save_figure(fig,name):
    fig.tight_layout()
    p=e06_out/'figures'/name
    fig.savefig(p,dpi=180,bbox_inches='tight')
    e06_figure_paths.append(str(p.relative_to(e06_out)))
    plt.show()
    plt.close(fig)

e06_fig,e06_ax=plt.subplots(figsize=(6.8,4.2))
e06_bars=e06_ax.bar(['Non-hateful (0)','Hateful (1)'],[int(e06_class.loc[e06_class.majority_label.eq(k),'examples'].sum()) for k in [0,1]],color=['#3479a6','#b75d56'])
e06_ax.bar_label(e06_bars);e06_ax.set_ylabel('Majority-vote examples');e06_ax.set_title('Label distribution: majority-vote candidates')
e06_save_figure(e06_fig,'01_class_distribution.png')
e06_fig,e06_axes=plt.subplots(1,2,figsize=(11,4.2))
for e06_ax,e06_col,e06_title in zip(e06_axes,['context_lexical_words','gloss_lexical_words'],['Context length','Linked gloss length']):
    e06_ax.hist(e06_metrics[e06_col],bins=30,color='#3479a6',edgecolor='white')
    e06_ax.set(xlabel='Regex lexical word count (not model tokens)',ylabel='Examples',title=e06_title)
e06_save_figure(e06_fig,'02_context_and_gloss_lengths.png')
e06_fig,e06_ax=plt.subplots(figsize=(6.8,4.2))
e06_ax.hist(e06_metrics.context_mean_word_characters.dropna(),bins=25,color='#538c77',edgecolor='white')
e06_ax.set(xlabel='Mean characters per regex lexical word',ylabel='Examples',title='Word lengths within context sentences')
e06_save_figure(e06_fig,'03_mean_word_length.png')
e06_fig,e06_ax=plt.subplots(figsize=(6.8,4.2))
e06_bars=e06_ax.bar(e06_annotation_counts.annotation_count.astype(str),e06_annotation_counts.examples,color='#3479a6')
e06_ax.bar_label(e06_bars);e06_ax.set(xlabel='Number of annotations per original example',ylabel='Original examples (log scale)',title='Annotation count audit');e06_ax.set_yscale('log')
e06_save_figure(e06_fig,'04_annotation_count_distribution.png')
print('Generated 4 summary figures; output folder:',str(e06_out.relative_to(e06_root)))

## 6. Save Tables and Verification Summary

Per-example exports contain IDs, lengths, labels and vote counts, without full sentences or annotator demographics. Frequency and term-support tables remain restricted research materials.

This step completes EDA. Target-position policy, unusual annotator counts, near duplicates, token alignment and data splits still require the later checks. EDA alone does not establish model results or acceptance of all F1–F4 deliverables.


In [ ]:
if 'e06_out' not in globals() or not (e06_out/'figures').is_dir():
    raise RuntimeError('Run the Section 5 plotting cell before saving.')
e06_tables = {
    'class_distribution.csv':e06_class,'length_summary.csv':e06_length_summary,
    'length_summary_by_label.csv':e06_by_class,'annotation_count_distribution.csv':e06_annotation_counts,
    'annotation_vote_audit.csv':e06_annotation_audit,'pos_label_counts.csv':e06_pos,
    'term_label_support.csv':e06_term_support,'sense_label_support.csv':e06_sense_support,
    'definition_coverage.csv':e06_coverage,'duplicate_summary.csv':e06_duplicate_summary,
    'duplicate_group_members.csv':e06_duplicates,'content_word_frequencies.csv':e06_content_frequency,
    'review_status_counts.csv':pd.DataFrame([{'decision':k,'examples':v} for k,v in sorted(e06_review_counts.items())])}
e06_metric_columns=['example_id','majority_label','annotation_count','has_gloss','has_sense_id']+e06_length_cols
e06_tables['example_length_metrics.csv']=e06_metrics[e06_metric_columns]
for e06_name,e06_table in e06_tables.items():
    e06_table.to_csv(e06_out/e06_name,index=False,encoding='utf-8-sig')
e06_protocol = {
    'schema_version':'hatewic-eda-v1','source_sha256':e06_review['source_sha256'],
    'input_sha256':e06_input_hashes,'raw_annotation_rows':len(e06_raw),
    'original_examples':len(e06_all),'majority_candidates':len(e06_master),
    'excluded_no_majority':len(e06_all)-len(e06_master),
    'review_counts':e06_review_counts,'python':sys.version,'platform':platform.platform(),'packages':e06_versions,
    'length_units':'Python characters; whitespace-separated items; Unicode regex lexical words. Not model tokens.',
    'content_token_pattern':r"[a-z]+(?:['’][a-z]+)*",'minimum_content_token_characters':2,
    'stopwords':sorted(e06_stopwords),'term_key':'NFKC+casefold+whitespace normalization, grouped jointly with POS',
    'duplicate_rules':['exact original text','NFKC+casefold+whitespace','same with punctuation ignored: candidate only'],
    'original_text_modified':False,'examples_deleted':False,'dataset_split_created':False,'model_trained':False,
    'definitions':'Dataset-linked gloss supplied with the example; no generated gloss, no automatic sense selection.',
    'subgroups':'Term/sense label mixing reported descriptively; hate-heterogeneous operational definition pending; training-unseen terms not computed.'}
(e06_out/'eda_protocol.json').write_text(json.dumps(e06_protocol,ensure_ascii=False,indent=2),encoding='utf-8')
for e06_p in e06_input_paths:
    if hashlib.sha256(e06_p.read_bytes()).hexdigest()!=e06_input_hashes[str(e06_p.relative_to(e06_root))]:
        raise RuntimeError('An input file changed during execution. Check the source before using these outputs.')
e06_note = '\n'.join([
    '# HateWiC Step 6 EDA results', '',
    f'Source annotation rows: {len(e06_raw)}; source examples: {len(e06_all)}; majority-label candidates: {len(e06_master)}。',
    f'Review status: {e06_review_counts}. Unresolved examples were not automatically excluded.',
    f'The analysis includes {e06_metrics.term.nunique()} source target terms and {len(e06_term_support)} normalized term and POS groups.',
    f'Term and POS groups with at least 2 examples per class: {int(e06_term_support.at_least_two_per_class.sum())} normalized term and POS groups.',
    '', 'Completed: input checks, class and length summaries, annotator-count and vote audits, gloss coverage, duplicate candidates, word-frequency tables and 4 figures.',
    'Pending at this stage: annotator-count policy, unresolved target positions, near-duplicate checks, model tokenization and truncation, final splits and training baselines.',
    'Length counts are not BERT token counts. Frequencies do not establish causation. Duplicate candidates are not automatically removed.',
    'Tables and figures share the same source hash. See eda_protocol.json for environment versions.',
    'Data and outputs remain restricted. This report does not contain model-performance or statistical-significance conclusions.',''])
(e06_out/'EDA_README.md').write_text(e06_note,encoding='utf-8')
e06_files = sorted(p for p in e06_out.rglob('*') if p.is_file())
e06_manifest = {str(p.relative_to(e06_out)):hashlib.sha256(p.read_bytes()).hexdigest() for p in e06_files}
(e06_out/'manifest.json').write_text(json.dumps(e06_manifest,indent=2),encoding='utf-8')
print('Step 6: saved EDA results verified')
print('Saved tables:',len(e06_tables),'; summary figures:',len(e06_figure_paths))
print('Input hashes unchanged; no examples removed, labels changed or review records overwritten.')
print('Output folder:',str(e06_out.relative_to(e06_root)))
print('Next: review target positions and unusual annotator counts, then check token alignment and prepare data splits.')

# HateWiC Step 7: Target-Position Ledger and Annotator-Count Policies

Build a per-example position-status ledger, list unusual annotator counts, and compare the sizes of alternative analysis sets. CPU is sufficient; no additional packages are needed.

Keep all majority-label candidates. Unique matches and repeated-surface rules remain candidate positions rather than manual confirmations. Preserve the statuses of the 91 uncertain and 9 unlocated cases. This section does not change labels, remove source examples or create a final training set.

## 1. Verify Sources and Summarise Positions

Run the next cell. If Step 6 variables are missing, rerun only Step 6 Sections 1 and 2 to mount Drive and read inputs. Do not repeat earlier reviews.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import pandas as pd

if not all(k in globals() for k in ['e06_root','e06_master','e06_all','e06_review','e06_input_hashes']):
    raise RuntimeError('Run Step 6 Sections 1 and 2 to mount Drive and read inputs first. Do not repeat Steps 1-5.')
e07_root=e06_root
e07_previous=e06_review_dir.parent
e07_paths=[e07_previous/'target_span_candidates_v03.csv',e07_previous/'review_summary.json',
           e06_review_dir/'rule_assigned_repeated_candidates.csv',e06_review_dir/'triage_policy.json']
e07_hashes=dict(e06_input_hashes)
for e07_rel,e07_digest in e06_input_hashes.items():
    if hashlib.sha256((e07_root/e07_rel).read_bytes()).hexdigest()!=e07_digest:
        raise ValueError('Inputs changed after Step 6. Rerun Step 6 Section 2 to check them; do not overwrite earlier reviews.')
for e07_path in e07_paths:
    if not e07_path.is_file():raise FileNotFoundError(f'Missing {e07_path.name}. Check the missing input; do not repeat the reviews.')
    e07_hashes[str(e07_path.relative_to(e07_root))]=hashlib.sha256(e07_path.read_bytes()).hexdigest()
e07_meta=json.loads(e07_paths[1].read_text())
e07_triage=json.loads(e07_paths[3].read_text())
if any(obj.get('source_sha256')!=e06_review['source_sha256'] for obj in [e07_meta,e07_triage]):
    raise ValueError('Step 3, Step 5 and source-file hashes differ.')
e07_candidates=pd.read_csv(e07_paths[0],keep_default_na=False,dtype={'example_id':str})
e07_rules=pd.read_csv(e07_paths[2],keep_default_na=False,dtype={'example_id':str})
if e07_candidates.example_id.duplicated().any() or set(e07_candidates.example_id)!=set(e06_master.example_id):
    raise ValueError('Step 3 IDs differ from the current majority-label table.')
if e07_rules.example_id.duplicated().any():raise ValueError('Duplicate rule-candidate IDs.')
e07_byid=e07_candidates.set_index('example_id')
e07_rulemap=e07_rules.set_index('example_id').to_dict(orient='index')
e07_records=e06_review['records']
if set(e07_rulemap)&set(e07_records):raise ValueError('Rule-assigned and manually reviewed IDs overlap.')
if not (set(e07_rulemap)|set(e07_records)).issubset(set(e07_byid.index)):
    raise ValueError('Review IDs include examples outside the main table.')
e07_rows=[]
for e07_row in e06_master.itertuples(index=False):
    e07_id=e07_row.example_id
    e07_old=e07_byid.loc[e07_id]
    e07_spans=json.loads(e07_old.candidate_spans)
    e07_start=e07_end=None
    if e07_id in e07_records:
        e07_item=e07_records[e07_id]
        e07_status='manual_'+e07_item['decision']
        if e07_item['decision']=='confirmed':
            e07_start,e07_end=e07_item['start'],e07_item['end']
            if e07_row.example[e07_start:e07_end]!=e07_item['surface']:
                raise ValueError('Manually confirmed span does not match the source text.')
        elif e07_item['decision'] not in ['uncertain','no_identifiable_target']:
            raise ValueError('Incomplete or unknown manual-review status.')
    elif e07_id in e07_rulemap:
        e07_rule=e07_rulemap[e07_id]
        if e07_old.alignment_status_03!='multiple_candidates_review':raise ValueError('Unexpected rule-triage source.')
        e07_saved=sorted([int(a),int(b)] for a,b in json.loads(e07_rule['all_candidate_spans']))
        if e07_saved!=sorted(e07_spans) or len(e07_saved)<2:raise ValueError('Repeated-surface rule candidates differ from Step 3.')
        if len({e07_row.example[a:b].casefold() for a,b in e07_saved})!=1:
            raise ValueError('Repeated candidates do not have the same surface form.')
        e07_start,e07_end=int(e07_rule['target_start_candidate']),int(e07_rule['target_end_candidate'])
        if [e07_start,e07_end]!=e07_saved[-1]:raise ValueError('Rule candidate does not follow the existing last-occurrence rule.')
        e07_status='repeated_same_surface_rule_candidate'
    else:
        if e07_old.alignment_status_03 not in ['unique_literal_candidate','unique_regular_form_candidate'] or len(e07_spans)!=1:
            raise ValueError('Multiple-candidate or unmatched items remain outside both the manual and rule queues.')
        e07_start,e07_end=e07_spans[0]
        e07_status=str(e07_old.alignment_status_03)
    for e07_a,e07_b in e07_spans:
        if not (isinstance(e07_a,int) and isinstance(e07_b,int) and 0<=e07_a<e07_b<=len(e07_row.example)):
            raise ValueError('Invalid Step 3 candidate-span type or range.')
    if e07_start is not None and not (0<=e07_start<e07_end<=len(e07_row.example)):
        raise ValueError('Ledger position is outside the source sentence.')
    e07_rows.append({'example_id':e07_id,'majority_label':e07_row.majority_label,
        'annotation_count':e07_row.annotation_count,'position_status':e07_status,
        'target_start_candidate':e07_start,'target_end_candidate':e07_end,
        'has_position_for_alignment_check':e07_start is not None,
        'manually_confirmed':e07_status=='manual_confirmed',
        'original_candidate_spans':json.dumps(e07_spans),
        'token_alignment_checked':False,'final_training_inclusion_decided':False})
e07_ledger=pd.DataFrame(e07_rows)
for e07_col in ['target_start_candidate','target_end_candidate']:
    e07_ledger[e07_col]=e07_ledger[e07_col].astype('Int64')
if len(e07_ledger)!=len(e06_master) or e07_ledger.example_id.duplicated().any():raise ValueError('Ledger counts do not match.')
e07_status_counts=e07_ledger.groupby('position_status').size().reset_index(name='examples')
print('Step 7: source and position-ledger checks passed')
print(e07_status_counts.to_string(index=False))
print('Total examples:',len(e07_ledger),'; unresolved target positions:',int((~e07_ledger.has_position_for_alignment_check).sum()))
print('A candidate position enables tokenization checks; it is not necessarily a gold span or an approved training example.')


## 2. Unusual Annotator Counts and Policy Impact

Run the next cell. List every source example with an annotator count other than three and its overlap with cases lacking a majority label or a resolved target position.

Use the majority-vote reconstruction verified earlier. Compare all majority-label candidates with a three-annotator-only subset. The latter is a proposed analysis policy, not an automatic exclusion rule. Whether high-count examples are control items requires evidence from the authors or data documentation.


In [ ]:
if 'e07_ledger' not in globals():raise RuntimeError('Run Step 7 Section 1 first.')
e07_vote_cols=['example_id','annotation_count','votes_non_hateful','votes_hateful','votes_cannot_decide','majority_label']
e07_special=e06_all.loc[e06_all.annotation_count.ne(3),e07_vote_cols].copy()
e07_special['in_majority_candidates']=e07_special.majority_label.notna()
e07_special=e07_special.merge(e07_ledger[['example_id','position_status','has_position_for_alignment_check']],
                             on='example_id',how='left',validate='one_to_one')
e07_special['source_role_verified']=False
e07_special['treatment_decided']=False
e07_pending=e07_ledger.loc[~e07_ledger.has_position_for_alignment_check].copy()
e07_scenarios=[]
for e07_name,e07_mask in [('all_majority_candidates',pd.Series(True,index=e07_ledger.index)),
                         ('three_annotators_only',e07_ledger.annotation_count.eq(3))]:
    for e07_posname,e07_posmask in [('all_position_statuses',pd.Series(True,index=e07_ledger.index)),
                                   ('position_candidate_available',e07_ledger.has_position_for_alignment_check)]:
        e07_subset=e07_ledger[e07_mask&e07_posmask]
        e07_scenarios.append({'annotation_scenario':e07_name,'position_scenario':e07_posname,
            'examples':len(e07_subset),'label_0':int(e07_subset.majority_label.eq(0).sum()),
            'label_1':int(e07_subset.majority_label.eq(1).sum()),'is_final_training_set':False})
e07_impact=pd.DataFrame(e07_scenarios)
print('Source examples with an annotator count other than 3:',len(e07_special),'; of these, with a majority label:',int(e07_special.in_majority_candidates.sum()))
print('Distribution of unusual annotator counts:')
print(e07_special.groupby('annotation_count').size().rename('examples').to_string())
print('Cohort sizes under alternative policies (no exclusions applied here):')
print(e07_impact.to_string(index=False))


## 3. Save Audit Lists and Decision Notes

Save position statuses, unusual annotator counts, unresolved positions, cohort sizes and class distributions under each policy, and the decisions still needed. Full sentences, glosses and annotator profiles are not displayed or exported by default. Outputs remain restricted to research use.

The next tokenization checks assess examples with candidate spans and retain a separate list of the others. Document and freeze final inclusion, duplicate grouping and split rules before formal training.


In [ ]:
if not all(k in globals() for k in ['e07_special','e07_impact']):raise RuntimeError('Complete the first two Step 7 cells before saving.')
for e07_rel,e07_digest in e07_hashes.items():
    if hashlib.sha256((e07_root/e07_rel).read_bytes()).hexdigest()!=e07_digest:
        raise ValueError('Input files changed. Saving stopped; check the source again.')
e07_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e07_out=e07_root/'results'/f'preparation_07_{e07_stamp}'
e07_out.mkdir(parents=True,exist_ok=False)
e07_tables={'position_status_ledger.csv':e07_ledger,'position_status_counts.csv':e07_status_counts,
    'non_three_annotation_review.csv':e07_special,'unresolved_position_ids.csv':e07_pending,
    'policy_impact_counts.csv':e07_impact}
for e07_name,e07_table in e07_tables.items():
    e07_table.to_csv(e07_out/e07_name,index=False,encoding='utf-8-sig')
e07_plan={'status':'audit_completed_policy_not_frozen','input_sha256':e07_hashes,
    'candidate_examples':len(e07_ledger),'unresolved_positions':len(e07_pending),
    'position_units':'Python characters, zero-based start, exclusive end',
    'non_three_role':'unverified; do not describe as quality-control items without source evidence',
    'non_three_recommendation':'Compare author-compatible inclusion with a strict-three sensitivity analysis; verify source before freezing.',
    'ambiguous_repeats':'Keep first/all/last candidate information for sensitivity analysis; existing last-position rule is not gold.',
    'unresolved_recommendation':'Retain all source rows. Define matched target-model evaluation coverage explicitly before training; do not fabricate spans.',
    'comparison_requirement':'Use identical final example IDs and partitions for matched models; report exclusion counts by label and term.',
    'next_checks':['tokenizer offsets and target coverage','context-preserving truncation','near-duplicate groups','freeze inclusion and split manifests'],
    'source_text_modified':False,'labels_modified':False,'rows_deleted':False,'split_created':False,'model_trained':False}
(e07_out/'preparation_protocol.json').write_text(json.dumps(e07_plan,ensure_ascii=False,indent=2),encoding='utf-8')
e07_note=f'# Step 7 preprocessing audit\nMajority-label candidates: {len(e07_ledger)}; positions available for technical checks: {int(e07_ledger.has_position_for_alignment_check.sum())}; unresolved target positions: {len(e07_pending)}.\nSource examples with an annotator count other than 3: {len(e07_special)}; of these, with a majority label: {int(e07_special.in_majority_candidates.sum())}.\nCandidate, manually confirmed and unresolved positions remain distinct. Final training inclusion and data splits have not been decided at this stage.\nConfirm the purpose of unusual annotator counts, the rule for repeated occurrences, and the analysis and sensitivity policy for unresolved positions.\nKeep the source data intact, use matched example sets across models, and report class and term coverage under each policy.\nNext, check subword alignment and truncation before fixing the splits. Label and position counts describe the audit, not model performance.\n'
(e07_out/'PREPARATION_README.md').write_text(e07_note,encoding='utf-8')
e07_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(e07_out.iterdir()) if p.is_file()}
(e07_out/'manifest.json').write_text(json.dumps(e07_manifest,indent=2),encoding='utf-8')
for e07_name,e07_digest in e07_manifest.items():
    if hashlib.sha256((e07_out/e07_name).read_bytes()).hexdigest()!=e07_digest:raise ValueError('Output verification failed.')
if pd.read_csv(e07_out/'position_status_ledger.csv').shape[0]!=len(e06_master):raise ValueError('Saved row count does not match.')
print('Step 7: saved results verified')
print('Output folder:',str(e07_out.relative_to(e07_root)))
print('Saved 5 tables, policy notes and file hashes. Source data and review records unchanged.')


# HateWiC Step 8: BERT and HateBERT Token Alignment and Truncation

Use the official fast tokenizers for both backbones. Download tokenizer and configuration files without model weights or training. CPU is sufficient; text is processed within this runtime.

Check model subword lengths for all majority-label candidates, character-to-subword alignment where a candidate span is available, right truncation versus target-preserving windows at lengths 128, 256 and 512, and the space needed for linked glosses.

Retain the Step 7 evidence levels: successful technical alignment does not establish the intended semantic position. Keep the 100 unresolved cases without inventing spans. Length settings are diagnostic; this step does not tune against test F1 or create final splits.

## 1. Prepare and Load Tokenizers

Run the next cell. The first run downloads public dependencies and tokenizer files and may take a few minutes. A GPU is unnecessary. Success displays `Step 8: tokenizers loaded`.

References: [Official HateBERT model](https://huggingface.co/GroNLP/hateBERT), [official BERT-base repository](https://huggingface.co/google-bert/bert-base-uncased), and [tokenizer and offset API](https://huggingface.co/docs/transformers/main_classes/tokenizer).


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import importlib, subprocess, sys, json, hashlib

try:e08_installed=version('transformers')
except PackageNotFoundError:e08_installed=None
if e08_installed!='4.57.1':
    if 'transformers' in sys.modules:
        raise RuntimeError('A different transformers version is already loaded. Check the version before continuing; do not run all cells or overwrite reviews.')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','transformers==4.57.1'])
    importlib.invalidate_caches()
from transformers import AutoTokenizer, AutoConfig
from huggingface_hub import snapshot_download

e08_root=Path('/content/drive/MyDrive/HateWiC_Project')
if not (e08_root/'data/raw/HateWiC_IndividualAnnos.csv').is_file():
    raise FileNotFoundError('Mount Drive using Step 6 Section 1 first. Do not rerun Steps 1-5.')
e08_specs={
 'bert_base':{'repo':'google-bert/bert-base-uncased','revision':'86b5e0934494bd15c9632b12f734a8a67f723594'},
 'hatebert':{'repo':'GroNLP/hateBERT','revision':'ab7a2d40f6c973cb57e63f23bfa2b51d981d028c'}}
e08_tokenizers={};e08_tokenizer_meta={}
for e08_name,e08_spec in e08_specs.items():
    e08_snapshot=Path(snapshot_download(e08_spec['repo'],revision=e08_spec['revision'],token=False,
        allow_patterns=['config.json','tokenizer_config.json','special_tokens_map.json','tokenizer.json','vocab.txt'],
        cache_dir=str(e08_root/'models/tokenizer_cache')))
    e08_tok=AutoTokenizer.from_pretrained(str(e08_snapshot),use_fast=True,local_files_only=True,trust_remote_code=False)
    e08_config=AutoConfig.from_pretrained(str(e08_snapshot),local_files_only=True,trust_remote_code=False)
    if not e08_tok.is_fast:raise ValueError('A fast tokenizer is required for source-text offsets.')
    if e08_config.max_position_embeddings<512:raise ValueError('Model position limit is incompatible with the diagnostic length.')
    if e08_tok.num_special_tokens_to_add(pair=False)!=2 or e08_tok.num_special_tokens_to_add(pair=True)!=3:
        raise ValueError('Special-token template differs from the expected BERT layout.')
    e08_tokenizers[e08_name]=e08_tok
    e08_tokenizer_meta[e08_name]={**e08_spec,'class':type(e08_tok).__name__,'is_fast':e08_tok.is_fast,
        'max_position_embeddings':e08_config.max_position_embeddings,
        'vocab_sha256':hashlib.sha256(json.dumps(e08_tok.get_vocab(),sort_keys=True).encode()).hexdigest(),
        'files_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in e08_snapshot.iterdir() if p.is_file()}}
e08_versions={p:version(p) for p in ['transformers','tokenizers','huggingface-hub','pandas','numpy']}
print('Step 8: tokenizers loaded')
print('Dependency versions:',e08_versions)
for e08_name,e08_meta in e08_tokenizer_meta.items():print(e08_name,e08_meta['class'],'; revision:',e08_meta['revision'])
print('Vocabulary hashes match:',e08_tokenizer_meta['bert_base']['vocab_sha256']==e08_tokenizer_meta['hatebert']['vocab_sha256'])
print('Loaded tokenizers and configurations; model weights have not been loaded.')


## 2. Read Step 7 Outputs and Test Alignment

Run the next cell. Verify the saved position ledger and source hashes. Use synthetic sentences to check repeated occurrences, multiword targets, inflections, case and targets at the end of long sentences. Manual reviews are not repeated.

If Step 6 or Step 7 variables are missing after a runtime reset, mount Drive, run Step 6 Section 2 and all three Step 7 cells, then return here.


In [ ]:
if not all(k in globals() for k in ['e08_tokenizers','e07_out','e07_hashes','e06_master']):
    raise RuntimeError('Load the tokenizers and restore Step 6 Section 2 and the three Step 7 cells. Do not repeat earlier reviews.')
e08_ledger_path=e07_out/'position_status_ledger.csv'
e08_manifest_path=e07_out/'manifest.json'
e08_saved_manifest=json.loads(e08_manifest_path.read_text())
if hashlib.sha256(e08_ledger_path.read_bytes()).hexdigest()!=e08_saved_manifest['position_status_ledger.csv']:
    raise ValueError('Saved Step 7 position-table hash mismatch.')
e08_input_hashes=dict(e07_hashes)
e08_input_hashes[str(e08_ledger_path.relative_to(e08_root))]=e08_saved_manifest['position_status_ledger.csv']
e08_input_hashes[str(e08_manifest_path.relative_to(e08_root))]=hashlib.sha256(e08_manifest_path.read_bytes()).hexdigest()
for e08_rel,e08_digest in e08_input_hashes.items():
    if hashlib.sha256((e08_root/e08_rel).read_bytes()).hexdigest()!=e08_digest:raise ValueError('Inputs changed. Check the source again.')
import pandas as pd
import unicodedata

e08_ledger=pd.read_csv(e08_ledger_path,keep_default_na=False,dtype={'example_id':str})
if e08_ledger.example_id.duplicated().any() or set(e08_ledger.example_id)!=set(e06_master.example_id):
    raise ValueError('Position-table IDs differ from the majority-label table.')
if not e08_ledger.has_position_for_alignment_check.isin([True,False]).all():raise ValueError('Position-availability flags are not Boolean.')
e08_data=e06_master[['example_id','example','term','pos','wiktionary_definition','majority_label']].merge(
    e08_ledger.drop(columns=['majority_label']),on='example_id',validate='one_to_one')

def e08_encode(tok,text):
    return tok(text,add_special_tokens=False,return_offsets_mapping=True,truncation=False,
               return_attention_mask=False,return_token_type_ids=False,verbose=False)

def e08_target_alignment(text,offsets,start,end):
    if not (isinstance(start,int) and isinstance(end,int) and 0<=start<end<=len(text)):
        raise ValueError('Invalid target character span.')
    indices=[i for i,(a,b) in enumerate(offsets) if b>a and a<end and b>start]
    required={i for i in range(start,end) if not text[i].isspace()}
    covered=set()
    for i in indices:
        a,b=offsets[i];covered.update(range(max(a,start),min(b,end)))
    spill=any(offsets[i][0]<start or offsets[i][1]>end for i in indices)
    complete=bool(indices) and bool(required) and required.issubset(covered)
    return indices,complete,spill

def e08_window(n,indices,budget):
    if not indices or indices[-1]-indices[0]+1>budget:return None
    left=max(0,min(indices[0]-(budget-(indices[-1]-indices[0]+1))//2,max(0,n-budget)))
    right=min(n,left+budget)
    if not (left<=indices[0]<=indices[-1]<right):raise ValueError('The window does not retain the target.')
    return left,right

def e08_check_prepared(tok,ids,indices,left,right,maxlen,gloss_ids=None):
    first=ids[left:right]
    prepared=tok.prepare_for_model(first,pair_ids=gloss_ids,add_special_tokens=True,truncation=False,
        return_special_tokens_mask=False,return_attention_mask=False,return_token_type_ids=False,verbose=False)
    expected=tok.build_inputs_with_special_tokens(first,gloss_ids)
    if prepared['input_ids']!=expected or len(expected)>maxlen:raise ValueError('Prepared input is too long or uses an unexpected template.')
    # In the BERT template, context follows CLS; target pooling excludes CLS and SEP.
    positions=[i-left+1 for i in indices]
    structural_mask=[1]+[0]*len(first)+[1]
    if gloss_ids is not None:structural_mask += [0]*len(gloss_ids)+[1]
    if len(structural_mask)!=len(expected):raise ValueError('Unexpected BERT structural-mask length.')
    if any(structural_mask[j] or expected[j]!=ids[i] for i,j in zip(indices,positions)):
        raise ValueError('Target subword positions do not match the special-token template.')
    return positions,len(expected)

for e08_tok in e08_tokenizers.values():
    for e08_s,e08_surface,e08_which in [('A cat meets another cat.','cat',1),
            ('The welfare queens arrived.','welfare queens',0),('Geezers spoke.','Geezers',0),
            ('The tanks were dipsticked.','dipsticked',0),('A naïve reader.','naïve',0)]:
        e08_start=[i for i in range(len(e08_s)) if e08_s.startswith(e08_surface,i)][e08_which]
        e08_enc=e08_encode(e08_tok,e08_s)
        e08_ix,e08_complete,e08_spill=e08_target_alignment(e08_s,e08_enc['offset_mapping'],e08_start,e08_start+len(e08_surface))
        if not e08_complete or e08_spill:raise ValueError('Synthetic target-alignment check failed.')
    e08_s='ordinary '*600+'final target'
    e08_start=e08_s.index('final target');e08_enc=e08_encode(e08_tok,e08_s)
    e08_ix,e08_complete,e08_spill=e08_target_alignment(e08_s,e08_enc['offset_mapping'],e08_start,len(e08_s))
    if e08_ix[-1]<126:raise ValueError('The long-sentence test did not create a right-truncation case.')
    e08_left,e08_right=e08_window(len(e08_enc['input_ids']),e08_ix,126)
    e08_check_prepared(e08_tok,e08_enc['input_ids'],e08_ix,e08_left,e08_right,128)
    e08_check_prepared(e08_tok,e08_enc['input_ids'],e08_ix,*e08_window(len(e08_enc['input_ids']),e08_ix,122),128,[100,101,102])
    if e08_window(300,list(range(200)),126) is not None:raise ValueError('An overlong target was not rejected.')
print('Step 8: input and synthetic alignment checks passed')
print('Majority-label candidates:',len(e08_data),'; with candidate positions:',int(e08_data.has_position_for_alignment_check.sum()))


## 3. Audit Full Tokenization, Alignment and Truncation

Run the next cell and wait for the summaries. Lengths 128/256/512 include special tokens. Right truncation retains the beginning of the context. Target-preserving windows centre on the target subwords and use available space near either boundary without rewriting the sentence.

Alignment must cover every non-whitespace target character without crossing its boundaries. Record boundary spill, missing coverage and unknown tokens separately. Slice windows directly from full-sentence token IDs rather than retokenizing sliced strings.

The gloss diagnostic retains at most 64 subwords from the linked gloss. Context budget equals total length minus 3 special tokens and the retained gloss length. This is a feasibility setting, not the final RO3 input policy. Matched models with and without glosses must use the same context window so a context change is not mistaken for a gloss benefit.

Linked-gloss conclusions are conditional on the relevant gloss being available at prediction time. This step does not evaluate automatic sense retrieval.


In [ ]:
if 'e08_data' not in globals():raise RuntimeError('Complete Step 8 Section 2 first.')
e08_lengths=(128,256,512);e08_gloss_cap=64
e08_metrics_rows=[];e08_window_rows=[];e08_previous_encodings={};e08_matches=[]
for e08_name,e08_tok in e08_tokenizers.items():
    for e08_row in e08_data.itertuples(index=False):
        e08_text=str(e08_row.example);e08_gloss=str(e08_row.wiktionary_definition)
        e08_enc=e08_encode(e08_tok,e08_text);e08_ids=e08_enc['input_ids'];e08_offsets=e08_enc['offset_mapping']
        e08_gids=e08_encode(e08_tok,e08_gloss)['input_ids'] if e08_gloss.strip() else []
        e08_n=len(e08_ids);e08_indices=[];e08_complete=False;e08_spill=False
        if e08_row.has_position_for_alignment_check:
            e08_start,e08_end=int(e08_row.target_start_candidate),int(e08_row.target_end_candidate)
            e08_indices,e08_complete,e08_spill=e08_target_alignment(e08_text,e08_offsets,e08_start,e08_end)
            e08_status='aligned' if e08_complete and not e08_spill else ('boundary_spill' if e08_spill else 'incomplete_character_coverage')
        else:e08_status='unresolved_character_position'
        e08_ok=e08_status=='aligned'
        e08_unk=any(e08_ids[i]==e08_tok.unk_token_id for i in e08_indices)
        e08_metrics_rows.append({'example_id':e08_row.example_id,'backbone':e08_name,
            'majority_label':e08_row.majority_label,'position_status':e08_row.position_status,
            'has_position_candidate':bool(e08_row.has_position_for_alignment_check),
            'alignment_status':e08_status,'context_subwords':e08_n,'context_tokens_with_specials':e08_n+2,
            'gloss_subwords':len(e08_gids),'pair_tokens_without_truncation':e08_n+len(e08_gids)+3,
            'target_subwords':len(e08_indices),'target_has_unk':e08_unk})
        e08_signature=(e08_ids,e08_offsets)
        if e08_name=='bert_base':e08_previous_encodings[e08_row.example_id]=e08_signature
        else:
            e08_matches.append({'example_id':e08_row.example_id,
                'same_input_ids':e08_previous_encodings[e08_row.example_id][0]==e08_ids,
                'same_offsets':e08_previous_encodings[e08_row.example_id][1]==e08_offsets})
        for e08_maxlen in e08_lengths:
            for e08_layout in ['context_only','context_plus_gloss']:
                e08_is_pair=e08_layout=='context_plus_gloss'
                e08_kept_gids=e08_gids[:e08_gloss_cap] if e08_is_pair else None
                e08_budget=e08_maxlen-(3 if e08_is_pair else 2)-(len(e08_kept_gids) if e08_is_pair else 0)
                e08_right_kept=(bool(e08_ok and e08_indices[-1]<e08_budget) if e08_indices else False)
                e08_win=e08_window(e08_n,e08_indices,e08_budget) if e08_ok else None
                e08_positions=[];e08_used=None;e08_left=e08_right=None
                if e08_win is not None:
                    e08_left,e08_right=e08_win
                    e08_positions,e08_used=e08_check_prepared(e08_tok,e08_ids,e08_indices,e08_left,e08_right,e08_maxlen,e08_kept_gids)
                e08_window_rows.append({'example_id':e08_row.example_id,'backbone':e08_name,
                    'majority_label':e08_row.majority_label,'layout':e08_layout,'max_length':e08_maxlen,
                    'alignment_ok':e08_ok,'position_status':e08_row.position_status,
                    'context_budget':e08_budget,'context_would_be_truncated':e08_n>e08_budget,
                    'gloss_available':bool(e08_gloss.strip()),
                    'gloss_retained_subwords':len(e08_kept_gids) if e08_is_pair else 0,
                    'gloss_truncated':bool(e08_is_pair and len(e08_gids)>e08_gloss_cap),
                    'right_truncation_keeps_complete_target':e08_right_kept,
                    'target_window_available':e08_win is not None,
                    'window_start_subword':e08_left,'window_end_subword_exclusive':e08_right,
                    'window_target_positions':json.dumps(e08_positions),'prepared_length':e08_used})
    print(e08_name,'tokenization checks complete:',len(e08_data))
e08_metrics=pd.DataFrame(e08_metrics_rows);e08_windows=pd.DataFrame(e08_window_rows);e08_equivalence=pd.DataFrame(e08_matches)
e08_alignment_summary=e08_metrics.groupby(['backbone','alignment_status']).size().reset_index(name='examples')
e08_summary=e08_windows.groupby(['backbone','layout','max_length']).agg(
    all_majority_examples=('example_id','size'),aligned_examples=('alignment_ok','sum'),
    context_truncated_examples=('context_would_be_truncated','sum'),
    right_keeps_target=('right_truncation_keeps_complete_target','sum'),
    target_window_available=('target_window_available','sum'),gloss_truncated_examples=('gloss_truncated','sum')).reset_index()
e08_summary['context_truncation_rate_all']=e08_summary.context_truncated_examples/e08_summary.all_majority_examples
for e08_col in ['right_keeps_target','target_window_available']:
    e08_summary[e08_col+'_rate_aligned']=e08_summary[e08_col]/e08_summary.aligned_examples.replace(0,float('nan'))
e08_summary['right_target_loss_among_aligned']=e08_summary.aligned_examples-e08_summary.right_keeps_target
e08_alignment_issues=e08_metrics[e08_metrics.alignment_status.ne('aligned')|e08_metrics.target_has_unk].copy()
if len(e08_metrics)!=2*len(e08_data) or len(e08_windows)!=12*len(e08_data):raise ValueError('Tokenization audit row count does not match.')
print('Step 8: full tokenization audit complete')
print(e08_alignment_summary.to_string(index=False))
print('Examples with identical input IDs from both tokenizers:',int(e08_equivalence.same_input_ids.sum()),'/',len(e08_equivalence))
print('Examples with identical offsets from both tokenizers:',int(e08_equivalence.same_offsets.sum()),'/',len(e08_equivalence))
print(e08_summary[['backbone','layout','max_length','context_truncated_examples','right_target_loss_among_aligned','target_window_available']].to_string(index=False))
print('Denominators: context truncation rates use all majority-label candidates; target retention rates use examples with valid character-to-subword alignment.')


## 4. Save Tokenization Checks

Save per-example length and alignment tables, window indices, issue IDs, model revisions and policy notes. Window indices remain technical candidates. Later training must reuse the implementation and fixed matched example sets. This step does not export sentences, glosses or complete input-token IDs, or create final train/validation/test partitions.


In [ ]:
if 'e08_summary' not in globals():raise RuntimeError('Complete Step 8 Section 3 first.')
for e08_rel,e08_digest in e08_input_hashes.items():
    if hashlib.sha256((e08_root/e08_rel).read_bytes()).hexdigest()!=e08_digest:raise ValueError('Inputs changed during the audit. Saving stopped.')
from datetime import datetime, timezone
e08_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e08_out=e08_root/'results'/f'tokenization_08_{e08_stamp}'
e08_out.mkdir(parents=True,exist_ok=False)
e08_tables={'tokenization_metrics.csv':e08_metrics,'window_diagnostics.csv':e08_windows,
 'alignment_summary.csv':e08_alignment_summary,'truncation_summary.csv':e08_summary,
 'alignment_issue_ids.csv':e08_alignment_issues,'tokenizer_equivalence.csv':e08_equivalence}
for e08_filename,e08_table in e08_tables.items():e08_table.to_csv(e08_out/e08_filename,index=False,encoding='utf-8-sig')
e08_protocol={'status':'technical_audit_only','input_sha256':e08_input_hashes,'versions':e08_versions,
 'tokenizers':e08_tokenizer_meta,'max_lengths_diagnosed':list(e08_lengths),'gloss_token_cap_diagnostic':e08_gloss_cap,
 'character_offsets':'zero based, end exclusive, unmodified original sentence',
 'alignment':'all non-whitespace target characters covered; reject token boundary spill; UNK flagged separately',
 'target_pooling_candidate':'mean of target subword hidden states, excluding special tokens; not executed here',
 'window_rule':'center around complete target subword interval; slice full-sentence token IDs; do not retokenize a substring',
 'gloss_ablation_requirement':'reuse exactly the same context window for with/without-gloss comparisons',
 'gloss_availability_assumption':'linked sense gloss provided; automatic sense retrieval not evaluated',
 'unresolved_positions_retained':True,'final_inclusion_frozen':False,'max_length_selected':False,
 'model_weights_loaded':False,'model_trained':False,'split_created':False,
 'references':['https://huggingface.co/GroNLP/hateBERT','https://huggingface.co/google-bert/bert-base-uncased',
               'https://huggingface.co/docs/transformers/main_classes/tokenizer']}
(e08_out/'tokenization_protocol.json').write_text(json.dumps(e08_protocol,ensure_ascii=False,indent=2),encoding='utf-8')
e08_readme='\n'.join(['# Step 8 tokenization and truncation checks',
 f'Majority-label candidates: {len(e08_data)}. Both backbones were checked on the same IDs.',
 'Alignment summary:',e08_alignment_summary.to_string(index=False),
 'Length and truncation summary:',e08_summary.to_string(index=False),
 'Truncation rates use all majority-label candidates; target retention rates use aligned examples.',
 'The 100 unresolved semantic positions are retained separately, not automatically removed as technical alignment failures.',
 'Candidate positions are not gold spans. Check unknown tokens separately. Gloss cap 64 and lengths 128/256/512 are diagnostic settings, not final parameters.',
 'Next: investigate alignment issues and near-duplicate groups, confirm cohort inclusion, fix train/validation/test IDs, then run baselines and models.',
 'This step has not loaded model weights, trained models, computed classification F1 or completed every foundation requirement.'])
(e08_out/'TOKENIZATION_README.md').write_text(e08_readme,encoding='utf-8')
e08_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(e08_out.iterdir()) if p.is_file()}
(e08_out/'manifest.json').write_text(json.dumps(e08_manifest,indent=2),encoding='utf-8')
for e08_name,e08_digest in e08_manifest.items():
    if hashlib.sha256((e08_out/e08_name).read_bytes()).hexdigest()!=e08_digest:raise ValueError('Saved output verification failed.')
print('Step 8: saved results verified')
print('Saved 6 audit tables, tokenizer settings, notes and file hashes.')
print('Output folder:',str(e08_out.relative_to(e08_root)))


# HateWiC Step 9: Near-Duplicate Candidates and Punctuation Boundaries

Prepare candidate groups for leakage checks and investigate the 1 boundary issue found in Step 8. CPU is sufficient.

Near duplicates are candidates, not necessarily the same annotated target occurrence. Review candidates with different labels. Related sentences across training and test partitions may inflate performance. Rare 5-character fragments do not identify every paraphrase; an empty search is not proof that leakage is absent.

## 1. Generate Near-Duplicate Candidate Groups

Run the next cell. Find identical normalized contexts and highly similar candidate pairs. Display counts without source sentences.


In [ ]:
from pathlib import Path
from collections import Counter,defaultdict
from itertools import combinations
import hashlib,json,re,unicodedata
import pandas as pd

if not all(k in globals() for k in ['e06_master','e08_input_hashes','e08_out','e08_metrics','e08_tokenizers']):
    raise RuntimeError('Restore Step 6 Section 2, the three Step 7 cells and Step 8 Sections 1-3. Completed manual reviews do not need to be repeated.')
e09_root=e08_root
for e09_rel,e09_digest in e08_input_hashes.items():
    if hashlib.sha256((e09_root/e09_rel).read_bytes()).hexdigest()!=e09_digest:
        raise ValueError('Input source changed. Check the files again.')
e09_source_hashes=dict(e08_input_hashes)
e09_source_hashes[str((e08_out/'tokenization_metrics.csv').relative_to(e09_root))]=hashlib.sha256((e08_out/'tokenization_metrics.csv').read_bytes()).hexdigest()
e09_data=e06_master[['example_id','example','majority_label']].copy()
e09_data.example_id=e09_data.example_id.astype(str)
if e09_data.example_id.duplicated().any():raise ValueError('Duplicate example IDs.')
def e09_norm(text):
    s=unicodedata.normalize('NFKC',str(text)).casefold()
    return ' '.join(re.sub(r'[^\w\s]',' ',s).split())
def e09_grams(s):return {s[i:i+5] for i in range(max(0,len(s)-4))}
e09_ids=e09_data.example_id.tolist();e09_labels=e09_data.majority_label.astype(int).tolist()
e09_norms=[e09_norm(x) for x in e09_data.example]
e09_sets=[e09_grams(x) for x in e09_norms]
e09_postings=defaultdict(list)
for e09_i,e09_gr in enumerate(e09_sets):
    for e09_g in e09_gr:e09_postings[e09_g].append(e09_i)
e09_candidate_pairs=set()
e09_by_norm=defaultdict(list)
for e09_i,e09_normtext in enumerate(e09_norms):e09_by_norm[e09_normtext].append(e09_i)
for e09_group in e09_by_norm.values():
    if len(e09_group)>1:e09_candidate_pairs.update(combinations(e09_group,2))
# Select at most 20 rare 5-character fragments per sentence; skip fragments shared by more than 80 sentences.
for e09_i,e09_gr in enumerate(e09_sets):
    e09_rare=sorted((g for g in e09_gr if 1<len(e09_postings[g])<=80),key=lambda g:(len(e09_postings[g]),g))[:20]
    for e09_g in e09_rare:
        for e09_j in e09_postings[e09_g]:
            if e09_j>e09_i:e09_candidate_pairs.add((e09_i,e09_j))
e09_pairs=[]
for e09_i,e09_j in sorted(e09_candidate_pairs):
    e09_a,e09_b=e09_sets[e09_i],e09_sets[e09_j]
    e09_exact=e09_norms[e09_i]==e09_norms[e09_j]
    if not e09_exact:
        if min(len(e09_norms[e09_i]),len(e09_norms[e09_j]))/max(1,max(len(e09_norms[e09_i]),len(e09_norms[e09_j])))<.8:continue
        e09_score=len(e09_a&e09_b)/len(e09_a|e09_b) if e09_a|e09_b else 0
        if e09_score<.85:continue
    else:e09_score=1.0
    e09_pairs.append({'example_id_a':e09_ids[e09_i],'example_id_b':e09_ids[e09_j],
        'normalized_exact':e09_exact,'char_5gram_jaccard':round(e09_score,6),
        'label_a':e09_labels[e09_i],'label_b':e09_labels[e09_j],
        'same_binary_label':e09_labels[e09_i]==e09_labels[e09_j]})
e09_pairs=pd.DataFrame(e09_pairs,columns=['example_id_a','example_id_b','normalized_exact',
    'char_5gram_jaccard','label_a','label_b','same_binary_label'])
e09_parent=list(range(len(e09_ids)))
def e09_find(i):
    while e09_parent[i]!=i:
        e09_parent[i]=e09_parent[e09_parent[i]];i=e09_parent[i]
    return i
def e09_union(i,j):
    a,b=e09_find(i),e09_find(j)
    if a!=b:e09_parent[max(a,b)]=min(a,b)
e09_lookup={v:i for i,v in enumerate(e09_ids)}
for e09_r in e09_pairs.itertuples(index=False):e09_union(e09_lookup[e09_r.example_id_a],e09_lookup[e09_r.example_id_b])
e09_groups=defaultdict(list)
for e09_i in range(len(e09_ids)):e09_groups[e09_find(e09_i)].append(e09_i)
e09_members=[]
for e09_group in e09_groups.values():
    if len(e09_group)<2:continue
    e09_group_id=min(e09_ids[i] for i in e09_group)
    for e09_i in e09_group:e09_members.append({'candidate_group_id':e09_group_id,
        'example_id':e09_ids[e09_i],'majority_label':e09_labels[e09_i],
        'group_size':len(e09_group),'label_conflict_in_group':len({e09_labels[j] for j in e09_group})>1})
e09_members=pd.DataFrame(e09_members,columns=['candidate_group_id','example_id','majority_label','group_size','label_conflict_in_group'])
print('Step 9: near-duplicate candidates generated')
print('Candidate pairs:',len(e09_pairs),'; candidate groups:',e09_members.candidate_group_id.nunique(),
      '; examples involved:',len(e09_members))
print('Candidate pairs with different labels:',int((~e09_pairs.same_binary_label).sum()),'; flagged for manual review.')
print('This screens near-duplicate candidates before splitting; final groups and partitions are not created at this step.')


## 2. Check Punctuation and Target Spans

Run the next cell. Identify Step 8 subwords that cross target boundaries and test whether **equal-length punctuation replacement** restores alignment. This diagnostic runs in memory; source sentences and reviewed spans remain unchanged. It does not automatically adopt the replacement for training.


In [ ]:
if 'e09_pairs' not in globals():raise RuntimeError('Run Step 9 Section 1 first.')
e09_spills=e08_metrics[e08_metrics.alignment_status.eq('boundary_spill')][['example_id','backbone']].copy()
e09_spills.example_id=e09_spills.example_id.astype(str)
e09_check_data=e08_data.set_index('example_id')
e09_cases=[]
e09_substitutions={'ǃ':'!'}
for e09_r in e09_spills.itertuples(index=False):
    e09_example=e09_check_data.loc[e09_r.example_id]
    e09_text=str(e09_example.example)
    e09_variant=''.join(e09_substitutions.get(ch,ch) for ch in e09_text)
    e09_tok=e08_tokenizers[e09_r.backbone]
    e09_encoded=e08_encode(e09_tok,e09_variant)
    e09_ix,e09_complete,e09_spill=e08_target_alignment(e09_variant,e09_encoded['offset_mapping'],
        int(e09_example.target_start_candidate),int(e09_example.target_end_candidate))
    e09_cases.append({'example_id':e09_r.example_id,'backbone':e09_r.backbone,
        'original_issue':'boundary_spill','contains_U01C3':('ǃ' in e09_text),
        'substitution_U01C3_to_ascii_exclamation_tested':e09_variant!=e09_text,
        'character_length_preserved':len(e09_variant)==len(e09_text),
        'variant_alignment_complete':bool(e09_complete and not e09_spill),
        'model_input_change_approved':False,'original_source_modified':False})
e09_cases=pd.DataFrame(e09_cases,columns=['example_id','backbone','original_issue','contains_U01C3',
    'substitution_U01C3_to_ascii_exclamation_tested','character_length_preserved',
    'variant_alignment_complete','model_input_change_approved','original_source_modified'])
print('Unique example IDs with Step 8 boundary issues:',e09_spills.example_id.nunique(),
      '; records across both tokenizers:',len(e09_cases))
print('Records with alignment restored after equal-length punctuation replacement:',int(e09_cases.variant_alignment_complete.sum()),
      '; source sentences and reviewed positions unchanged.')
print('Decide and document whether to use this normalization in model inputs before freezing the training configuration.')


## 3. Save Candidate Groups and Audit Notes

Save candidate pairs, group IDs, punctuation diagnostics and method notes. Tables contain example IDs, labels and statistics rather than source sentences. Group constraints will be applied in the later splitting step; this section does not create final partitions.


In [ ]:
if 'e09_cases' not in globals():raise RuntimeError('Complete the first two Step 9 cells first.')
for e09_rel,e09_digest in e09_source_hashes.items():
    if hashlib.sha256((e09_root/e09_rel).read_bytes()).hexdigest()!=e09_digest:
        raise ValueError('Inputs changed. Saving stopped.')
from datetime import datetime,timezone
e09_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e09_out=e09_root/'results'/f'duplicate_audit_09_{e09_stamp}'
e09_out.mkdir(parents=True,exist_ok=False)
e09_tables={'near_duplicate_candidate_pairs.csv':e09_pairs,
 'near_duplicate_candidate_groups.csv':e09_members,
 'token_boundary_exception.csv':e09_cases}
for e09_name,e09_table in e09_tables.items():e09_table.to_csv(e09_out/e09_name,index=False,encoding='utf-8-sig')
e09_protocol={'status':'candidate_screening_only','input_sha256':e09_source_hashes,
 'normalized_text':'NFKC + casefold + punctuation-to-space + whitespace collapse',
 'candidate_generation':'all exact-normalized pairs plus up to 20 rare character 5grams per sentence; postings with 2-80 examples',
 'acceptance_for_review':'length ratio >= 0.8 and 5gram Jaccard >= 0.85; exact normalized pairs always included',
 'candidate_coverage':'not exhaustive for all semantic paraphrases; grouping by connected components',
 'label_conflicts':'flag for review; no automatic label change',
 'model_input_normalization_trial':'U+01C3 punctuation replaced by one ASCII ! in memory; original text and offsets unchanged; not approved for training',
 'dataset_split_created':False,'rows_deleted':False,'labels_changed':False}
(e09_out/'duplicate_audit_protocol.json').write_text(json.dumps(e09_protocol,ensure_ascii=False,indent=2),encoding='utf-8')
e09_note='\n'.join(['# Step 9 audit results',f'Near-duplicate candidate pairs: {len(e09_pairs)}; candidate groups: {e09_members.candidate_group_id.nunique()}; examples involved: {len(e09_members)}。',
 f'Candidate pairs with different labels: {int((~e09_pairs.same_binary_label).sum())}; manual review required.',
 f'Example IDs with punctuation-boundary issues: {e09_spills.example_id.nunique()}; tokenizer records restored by equal-length punctuation replacement: {int(e09_cases.variant_alignment_complete.sum())}/{len(e09_cases)}。',
 'These are candidates and diagnostics. Source sentences, reviewed spans, hate labels and final partitions have not been changed.',
 'Next: confirm input preprocessing and inclusion policy, keep confirmed near-duplicate groups in one partition, save fixed IDs and seeds, then run baselines.'])
(e09_out/'DUPLICATE_AUDIT_README.md').write_text(e09_note,encoding='utf-8')
e09_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(e09_out.iterdir()) if p.is_file()}
(e09_out/'manifest.json').write_text(json.dumps(e09_manifest,indent=2),encoding='utf-8')
for e09_name,e09_digest in e09_manifest.items():
    if hashlib.sha256((e09_out/e09_name).read_bytes()).hexdigest()!=e09_digest:raise ValueError('Output verification failed.')
print('Step 9: saved results verified')
print('Output folder:',str(e09_out.relative_to(e09_root)))
print('Source data, reviews and tokenization results retained.')


# HateWiC Step 10: Matched Analysis Set and Reproducible Split Draft

Prepare a **provisional split for supervisor confirmation**. Matched BERT/HateBERT comparisons must use the same example IDs and partitions. CPU is sufficient.

The draft includes 3,745 of 3,845 majority-label examples with an available target-position candidate. List the 100 unresolved cases and their reasons separately. The included set contains manual confirmations, unique candidates and repeated-surface rule candidates, with evidence levels retained. For example ID 1707, use the equal-length punctuation replacement in derived model input; preserve the source sentence. Include unusual annotator counts provisionally where a majority label and position are available, and provide a three-annotator-only sensitivity list.

Use approximate proportions of 70/15/15. Keep the duplicate-sentence candidate groups from Step 9 in one partition. Other paraphrases may remain undetected. Define unseen terms and terms with both labels using training data only.

This step creates **draft ID lists**, without training or tuning on test results. Confirm inclusion and repeated-occurrence policies with the supervisor before formal experiments, then use the same IDs across all matched models.

## 1. Verify Step 9 Inputs

Run the next cell.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime,timezone
import hashlib,json,unicodedata
import numpy as np
import pandas as pd

if not all(k in globals() for k in ['e09_out','e09_pairs','e09_members','e09_cases','e08_data','e08_metrics','e07_ledger','e06_master']):
    raise RuntimeError('Restore Step 6 Section 2, all Step 7 cells, Step 8 Sections 1-3 and all Step 9 cells. Do not repeat manual reviews.')
e10_root=e09_root
e10_manifest=json.loads((e09_out/'manifest.json').read_text())
for e10_name,e10_digest in e10_manifest.items():
    if hashlib.sha256((e09_out/e10_name).read_bytes()).hexdigest()!=e10_digest:
        raise ValueError('Step 9 output hash mismatch.')
e10_inputs=dict(e09_source_hashes)
e10_inputs[str((e09_out/'manifest.json').relative_to(e10_root))]=hashlib.sha256((e09_out/'manifest.json').read_bytes()).hexdigest()
for e10_rel,e10_digest in e10_inputs.items():
    if hashlib.sha256((e10_root/e10_rel).read_bytes()).hexdigest()!=e10_digest:
        raise ValueError('Inputs changed. Verify Step 9 outputs again.')
e10_position=e07_ledger[['example_id','annotation_count','position_status',
    'has_position_for_alignment_check','target_start_candidate','target_end_candidate']].copy()
e10_position.example_id=e10_position.example_id.astype(str)
e10_tokens=e08_metrics.loc[e08_metrics.backbone.eq('bert_base'),['example_id','alignment_status']].copy()
e10_tokens.example_id=e10_tokens.example_id.astype(str)
e10_base=e06_master[['example_id','term','pos','majority_label','wiktionary_definition']].copy()
e10_base.example_id=e10_base.example_id.astype(str)
e10_base=e10_base.merge(e10_position,on='example_id',validate='one_to_one').merge(e10_tokens,on='example_id',validate='one_to_one')
if len(e10_base)!=3845 or e10_base.example_id.duplicated().any():raise ValueError('Invalid example-table IDs.')
print('Step 10: input checks passed;',len(e10_base),'majority-label candidates.')


## 2. Build the Matched Target-Model Cohort

Run the next cell. List the 100 examples without a resolved position separately. Apply the **equal-length input normalization** tested in Step 9 and retain an exception flag; source files stay unchanged. Set the draft cohort without selecting examples according to model performance.


In [ ]:
if 'e10_base' not in globals():raise RuntimeError('Run Step 10 Section 1 first.')
e10_exception_ids=set(e09_cases.loc[e09_cases.variant_alignment_complete,'example_id'].astype(str))
if e10_exception_ids!={'1707'} or not e09_cases.variant_alignment_complete.all():
    raise ValueError('Punctuation diagnostic status changed. The current draft cannot be applied automatically.')
e10_base['technical_target_ready']=e10_base.alignment_status.eq('aligned')|e10_base.example_id.isin(e10_exception_ids)
e10_base['needs_U01C3_input_substitution']=e10_base.example_id.isin(e10_exception_ids)
if not e10_base.loc[e10_base.needs_U01C3_input_substitution,'has_position_for_alignment_check'].all():
    raise ValueError('The exception example has no manually confirmed position.')
e10_included=e10_base[e10_base.technical_target_ready].copy().reset_index(drop=True)
e10_excluded=e10_base[~e10_base.technical_target_ready][['example_id','majority_label','annotation_count','position_status','alignment_status']].copy()
e10_excluded['exclusion_reason']='no_resolved_target_character_position'
if len(e10_included)!=3745 or len(e10_excluded)!=100:
    raise ValueError('Draft cohort size differs from the preceding audit. Check the source; do not change the expected count to force a match.')
if set(e10_included.example_id)&set(e10_excluded.example_id):raise ValueError('Included and excluded IDs overlap.')
if not e10_included.target_start_candidate.notna().all() or not e10_included.target_end_candidate.notna().all():
    raise ValueError('The draft still contains missing target positions.')
e10_norm=lambda s:' '.join(unicodedata.normalize('NFKC',str(s)).casefold().split())
e10_included['term_pos_key']=[hashlib.sha256((e10_norm(t)+'\x1f'+e10_norm(p)).encode()).hexdigest()
    for t,p in zip(e10_included.term,e10_included.pos)]
e10_included['linked_gloss_available']=e10_included.wiktionary_definition.astype(str).str.strip().ne('')
e10_group_map=dict(zip(e09_members.example_id.astype(str),e09_members.candidate_group_id.astype(str)))
e10_included['split_group_id']=[e10_group_map.get(k,k) for k in e10_included.example_id]
e10_included['three_annotators']=e10_included.annotation_count.eq(3)
print('Provisional target-model cohort:',len(e10_included),'; unresolved target positions:',len(e10_excluded))
print('Draft-cohort examples with an annotator count other than 3:',int((~e10_included.three_annotators).sum()),
      '; examples requiring equal-length punctuation normalization:',int(e10_included.needs_U01C3_input_substitution.sum()))
print('No source examples removed, labels changed or models trained at this step.')


## 3. Generate Fixed Partitions and Check Leakage

Run the next cell. Assign complete duplicate-candidate groups using a fixed seed, approximating the 70/15/15 size ratio and class balance. Check disjoint IDs, group containment, class counts and training-unseen terms. If a test subgroup lacks a true class, mark its later macro F1 as not estimable.


In [ ]:
if 'e10_included' not in globals():raise RuntimeError('Run Step 10 Section 2 first.')
e10_seed=20261021
e10_folds=['train','validation','test']
e10_fracs={'train':.70,'validation':.15,'test':.15}
e10_counts=e10_included.majority_label.value_counts().to_dict()
e10_target_total={s:len(e10_included)*f for s,f in e10_fracs.items()}
e10_target_label={s:{k:e10_counts.get(k,0)*f for k in [0,1]} for s,f in e10_fracs.items()}
e10_groups=[]
for e10_gid,e10_g in e10_included.groupby('split_group_id',sort=False):
    e10_c=e10_g.majority_label.value_counts().to_dict()
    e10_groups.append({'id':str(e10_gid),'ids':e10_g.example_id.tolist(),'n':len(e10_g),
        'labels':{k:int(e10_c.get(k,0)) for k in [0,1]}})
e10_rng=np.random.default_rng(e10_seed)
e10_rng.shuffle(e10_groups)
e10_groups.sort(key=lambda g:-g['n'])
e10_current={s:{'n':0,0:0,1:0} for s in e10_folds}
e10_assign={}
def e10_cost(s,g):
    cur=e10_current[s];n=cur['n']+g['n']
    score=(n-e10_target_total[s])**2/max(1,e10_target_total[s])
    for k in [0,1]:score+=(cur[k]+g['labels'][k]-e10_target_label[s][k])**2/max(1,e10_target_label[s][k])
    return score
for e10_group in e10_groups:
    e10_best=min(e10_folds,key=lambda s:(e10_cost(s,e10_group)-e10_cost(s,{'n':0,'labels':{0:0,1:0}}),
                                          e10_current[s]['n']/max(1,e10_target_total[s]),s))
    e10_current[e10_best]['n']+=e10_group['n']
    for e10_k in [0,1]:e10_current[e10_best][e10_k]+=e10_group['labels'][e10_k]
    for e10_id in e10_group['ids']:e10_assign[e10_id]=e10_best
e10_included['split']=e10_included.example_id.map(e10_assign)
if e10_included.split.isna().any() or len(e10_assign)!=len(e10_included):raise ValueError('Some examples are missing from the split.')
if e10_included.groupby('split_group_id').split.nunique().gt(1).any():raise ValueError('A near-duplicate candidate group crosses training and test partitions.')
for e10_s in e10_folds:
    e10_actual=int(e10_included.split.eq(e10_s).sum())
    if abs(e10_actual-e10_target_total[e10_s])>max(10,.02*len(e10_included)):
        raise ValueError('Split proportions differ from the expected range. Check the grouping.')
e10_train=e10_included[e10_included.split.eq('train')]
e10_train_terms=set(e10_train.term_pos_key)
e10_hetero=(e10_train.groupby('term_pos_key').majority_label.nunique().eq(2))
e10_hetero_terms=set(e10_hetero[e10_hetero].index)
e10_included['term_seen_in_train']=e10_included.term_pos_key.isin(e10_train_terms)
e10_included['hate_heterogeneous_in_train']=e10_included.term_pos_key.isin(e10_hetero_terms)
e10_included['split_group_size']=e10_included.groupby('split_group_id').example_id.transform('size')
e10_summary=e10_included.groupby(['split','majority_label']).size().unstack(fill_value=0).reindex(e10_folds)
e10_summary.columns=['label_'+str(c) for c in e10_summary.columns]
e10_summary['total']=e10_summary.sum(axis=1)
e10_subgroups=[]
for e10_s in ['validation','test']:
    for e10_kind,e10_mask in [('training_unseen_term',~e10_included.term_seen_in_train),
                             ('hate_heterogeneous_train_term',e10_included.hate_heterogeneous_in_train)]:
        e10_sub=e10_included[e10_included.split.eq(e10_s)&e10_mask]
        e10_subgroups.append({'split':e10_s,'subgroup':e10_kind,'examples':len(e10_sub),
            'label_0':int(e10_sub.majority_label.eq(0).sum()),
            'label_1':int(e10_sub.majority_label.eq(1).sum()),
            'macro_f1_estimable_with_both_labels':e10_sub.majority_label.nunique()==2})
e10_subgroups=pd.DataFrame(e10_subgroups)
print('Step 10: split and leakage checks passed')
print(e10_summary.to_string())
print('Evaluation subgroups defined from training data:')
print(e10_subgroups.to_string(index=False))
print('All near-duplicate candidate groups stay within one partition. Models have not yet been trained.')


## 4. Save the Split Draft and Evidence

Run the next cell. Save analysis IDs, spans, partition assignments, the 100 excluded target-model IDs and reasons, training-defined subgroups and policy notes. **These outputs remain provisional.** If annotator-count, position or repeated-word policies change, regenerate and version the split before training. Do not adjust inclusion according to test-model scores.


In [ ]:
if 'e10_summary' not in globals():raise RuntimeError('Complete the first three Step 10 cells first.')
for e10_rel,e10_digest in e10_inputs.items():
    if hashlib.sha256((e10_root/e10_rel).read_bytes()).hexdigest()!=e10_digest:raise ValueError('Inputs changed. Saving stopped.')
e10_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e10_out=e10_root/'results'/f'split_draft_10_{e10_stamp}'
e10_out.mkdir(parents=True,exist_ok=False)
e10_cols=['example_id','split','majority_label','annotation_count','three_annotators','position_status',
 'target_start_candidate','target_end_candidate','needs_U01C3_input_substitution',
 'split_group_id','split_group_size','term_pos_key','term_seen_in_train',
 'hate_heterogeneous_in_train','linked_gloss_available']
e10_splits=e10_included[e10_cols].sort_values('example_id',key=lambda c:c.astype(int)).reset_index(drop=True)
e10_trainonly_sensitivity=e10_splits[e10_splits.three_annotators].copy()
e10_cross_groups=e09_members[e09_members.label_conflict_in_group].copy()
e10_tables={'split_draft_ids.csv':e10_splits,
 'target_model_unresolved_exclusions.csv':e10_excluded,
 'three_annotator_sensitivity_ids.csv':e10_trainonly_sensitivity,
 'split_class_counts.csv':e10_summary.reset_index(),
 'training_defined_subgroup_counts.csv':e10_subgroups,
 'cross_label_duplicate_candidate_groups.csv':e10_cross_groups}
for e10_name,e10_table in e10_tables.items():e10_table.to_csv(e10_out/e10_name,index=False,encoding='utf-8-sig')
e10_protocol={'status':'provisional_split_draft_before_model_training',
 'raw_source_sha256':e06_review['source_sha256'],'input_sha256':e10_inputs,
 'seed':e10_seed,'fractions':e10_fracs,'split_grouping':'connected components of ninth-step same/near-text candidate groups; group kept in one split',
 'model_comparison_requirement':'same included IDs and same train/validation/test split for every matched target model and baseline',
 'included_position_policy':'use manually confirmed or prior recorded unique/repeated candidate spans; candidate quality status retained',
 'excluded_reason':'unresolved target position; original rows and labels retained outside target-model cohort',
 'rare_punctuation_policy':'for ID1707 model input only, replace U+01C3 with same-length ASCII !; original example and offsets unchanged; apply identically across model backbones',
 'non_three_policy':'included when majority and location are available; separate three-annotator-only sensitivity list',
 'term_key':'SHA256(NFKC+casefold+whitespace normalized term, POS); seen/heterogeneous sets calculated from training IDs only',
 'gloss_assumption':'linked gloss available as given; automatic retrieval not evaluated',
 'matching_limit':'rare 5-gram candidate screen does not exhaust all paraphrases',
 'formal_protocol_requires_advisor_confirmation':True,'model_trained':False,'test_metric_inspected':False}
(e10_out/'split_draft_protocol.json').write_text(json.dumps(e10_protocol,ensure_ascii=False,indent=2),encoding='utf-8')
e10_readme='\n'.join(['# Step 10 matched data split draft',
 f'Original majority-label candidates: {len(e10_base)}; provisional target-model cohort: {len(e10_splits)}; unresolved target positions: {len(e10_excluded)}。',
 'Partition counts:',e10_summary.to_string(),
 'Subgroups defined from training data:',e10_subgroups.to_string(index=False),
 f'Included examples with an annotator count other than 3: {int((~e10_splits.three_annotators).sum())}; punctuation-normalization candidates: {int(e10_splits.needs_U01C3_input_substitution.sum())}。',
 'Duplicate-sentence candidate groups stay in one partition. All matched models and baselines must use the same ID lists.',
 'This draft requires supervisor confirmation of candidate positions, unusual annotator counts and punctuation preprocessing before formal training.',
 'Models have not been trained or tuned using test results.'])
(e10_out/'SPLIT_DRAFT_README.md').write_text(e10_readme,encoding='utf-8')
e10_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(e10_out.iterdir()) if p.is_file()}
(e10_out/'manifest.json').write_text(json.dumps(e10_manifest,indent=2),encoding='utf-8')
for e10_name,e10_digest in e10_manifest.items():
    if hashlib.sha256((e10_out/e10_name).read_bytes()).hexdigest()!=e10_digest:raise ValueError('Output hash verification failed.')
e10_check=pd.read_csv(e10_out/'split_draft_ids.csv',dtype={'example_id':str})
if len(e10_check)!=len(e10_splits) or e10_check.example_id.duplicated().any():raise ValueError('Saved split IDs are incomplete.')
if e10_check.groupby('split_group_id').split.nunique().gt(1).any():raise ValueError('A saved group crosses partitions.')
print('Step 10: saved results verified')
print('Saved cohort IDs, exclusion and sensitivity lists, subgroup counts, protocol and hashes.')
print('Output folder:',str(e10_out.relative_to(e10_root)))
print('This is a provisional split for supervisor review; model experiments remain to be completed.')


# HateWiC Step 11: Training and Validation Baselines

Use the same draft IDs from Step 10 for a majority baseline, target-word prior and context TF-IDF with logistic regression. Fit models and estimate priors using training data only. Evaluate on validation IDs; reserve test evaluation until preprocessing and model configurations are fixed.

These baselines check the data pipeline and do not represent BERT or HateBERT performance. Inputs are source context or training-term statistics, without linked glosses. CPU is sufficient.

Report per-class F1, macro F1, balanced accuracy and confusion matrices. Define training-unseen terms and terms with both labels from training data. Mark subgroup macro F1 as not estimable if a true class is absent.

## 1. Verify the Draft and Prepare Validation Inputs

Run the next cell.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
from importlib.metadata import version,PackageNotFoundError
import hashlib,json,unicodedata
import numpy as np
import pandas as pd

if not all(k in globals() for k in ['e10_out','e10_splits','e06_master','e10_inputs']):
    raise RuntimeError('Restore Step 6 Section 2, all Step 7 cells, Step 8 Sections 1-3, all Step 9 cells and all Step 10 cells. Do not repeat manual reviews.')
e11_root=e10_root
e11_manifest=json.loads((e10_out/'manifest.json').read_text())
for e11_name,e11_digest in e11_manifest.items():
    if hashlib.sha256((e10_out/e11_name).read_bytes()).hexdigest()!=e11_digest:
        raise ValueError('Step 10 split-file hash mismatch.')
e11_inputs=dict(e10_inputs)
e11_inputs[str((e10_out/'split_draft_ids.csv').relative_to(e11_root))]=e11_manifest['split_draft_ids.csv']
e11_inputs[str((e10_out/'manifest.json').relative_to(e11_root))]=hashlib.sha256((e10_out/'manifest.json').read_bytes()).hexdigest()
for e11_rel,e11_digest in e11_inputs.items():
    if hashlib.sha256((e11_root/e11_rel).read_bytes()).hexdigest()!=e11_digest:raise ValueError('Input files changed.')
e11_split=pd.read_csv(e10_out/'split_draft_ids.csv',keep_default_na=False,dtype={'example_id':str,'term_pos_key':str,'split_group_id':str})
if set(e11_split.example_id)!=set(e10_splits.example_id) or e11_split.example_id.duplicated().any():
    raise ValueError('Split IDs differ from the Step 10 results in memory.')
e11_text=e06_master[['example_id','example']].copy();e11_text.example_id=e11_text.example_id.astype(str)
e11_data=e11_split.merge(e11_text,on='example_id',validate='one_to_one')
if len(e11_data)!=3745:raise ValueError('Draft and source-text tables did not match completely.')
e11_train=e11_data[e11_data.split.eq('train')].copy().reset_index(drop=True)
e11_val=e11_data[e11_data.split.eq('validation')].copy().reset_index(drop=True)
if len(e11_train)!=2622 or len(e11_val)!=562:raise ValueError('Partition sizes differ from the preceding audit.')
if set(e11_train.example_id)&set(e11_val.example_id):raise ValueError('Training and validation IDs overlap.')
if set(e11_train.split_group_id)&set(e11_val.split_group_id):raise ValueError('A duplicate candidate group crosses training and validation partitions.')
def e11_model_text(row):
    text=str(row.example)
    return text.replace('ǃ','!') if row.needs_U01C3_input_substitution else text
e11_train_text=[e11_model_text(r) for r in e11_train.itertuples(index=False)]
e11_val_text=[e11_model_text(r) for r in e11_val.itertuples(index=False)]
print('Step 11: input checks passed; training:',len(e11_train),'; validation:',len(e11_val))
print('The Step 10 test partition was not used for fitting or evaluation.')


## 2. Train Three Baselines and Calculate Validation Metrics

Run the next cell. Determine the majority class from training labels. Estimate term-and-POS priors from training counts, using the training majority for ties and unseen terms. Fit the TF-IDF vocabulary and IDF on training contexts only. Logistic regression uses fixed parameters rather than repeated validation/test-based tuning. Results support pipeline checks and later matched comparisons.

If scikit-learn is already loaded, use and record that live version. Otherwise install version 1.7.2. Use the saved configuration to reproduce the run.


In [ ]:
if 'e11_train' not in globals():raise RuntimeError('Complete Step 11 Section 1 first.')
import sys,subprocess,importlib
if 'sklearn' not in sys.modules:
    try:e11_sklearn_version=version('scikit-learn')
    except PackageNotFoundError:e11_sklearn_version=None
    if e11_sklearn_version!='1.7.2':
        subprocess.check_call([sys.executable,'-m','pip','install','--quiet','scikit-learn==1.7.2'])
        importlib.invalidate_caches()
import sklearn
e11_sklearn_version=sklearn.__version__
print('scikit-learn version used for this baseline run:',e11_sklearn_version)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score,balanced_accuracy_score,confusion_matrix

e11_ytrain=e11_train.majority_label.to_numpy(dtype=int)
e11_yval=e11_val.majority_label.to_numpy(dtype=int)
if set(e11_ytrain)!={0,1} or set(e11_yval)!={0,1}:raise ValueError('Training or validation data are missing a class.')
e11_majority=int(pd.Series(e11_ytrain).value_counts().sort_index().idxmax())
e11_predictions={'train_majority':np.repeat(e11_majority,len(e11_val))}
e11_train_keycounts=e11_train.groupby('term_pos_key').majority_label.value_counts().unstack(fill_value=0)
e11_predictions['training_term_prior']=np.array([
    (int(e11_train_keycounts.loc[k,1]>e11_train_keycounts.loc[k,0]) if k in e11_train_keycounts.index
     else e11_majority) for k in e11_val.term_pos_key],dtype=int)
e11_vectorizer=TfidfVectorizer(lowercase=True,strip_accents='unicode',ngram_range=(1,2),
    min_df=2,max_features=50000,sublinear_tf=True,token_pattern=r'(?u)\b\w+\b')
e11_xtrain=e11_vectorizer.fit_transform(e11_train_text)
e11_xval=e11_vectorizer.transform(e11_val_text)
e11_classifier=LogisticRegression(C=1.0,class_weight='balanced',max_iter=1000,solver='lbfgs',random_state=20261021)
e11_classifier.fit(e11_xtrain,e11_ytrain)
e11_predictions['context_tfidf_logreg']=e11_classifier.predict(e11_xval)
if set(e11_classifier.classes_)!={0,1}:raise ValueError('Unexpected classifier class order.')
e11_prediction_table=e11_val[['example_id','majority_label','three_annotators','term_seen_in_train',
    'hate_heterogeneous_in_train','position_status']].copy()
for e11_name,e11_pred in e11_predictions.items():
    if len(e11_pred)!=len(e11_val) or not set(e11_pred).issubset({0,1}):raise ValueError('Invalid prediction table.')
    e11_prediction_table[e11_name]=e11_pred
e11_metrics=[]
for e11_name,e11_pred in e11_predictions.items():
    e11_f1=f1_score(e11_yval,e11_pred,labels=[0,1],average=None,zero_division=0)
    e11_cm=confusion_matrix(e11_yval,e11_pred,labels=[0,1])
    e11_metrics.append({'model':e11_name,'split':'validation','examples':len(e11_val),
        'macro_f1':float(np.mean(e11_f1)),'f1_label_0':float(e11_f1[0]),
        'f1_label_1':float(e11_f1[1]),'balanced_accuracy':float(balanced_accuracy_score(e11_yval,e11_pred)),
        'true0_pred0':int(e11_cm[0,0]),'true0_pred1':int(e11_cm[0,1]),
        'true1_pred0':int(e11_cm[1,0]),'true1_pred1':int(e11_cm[1,1])})
e11_metrics=pd.DataFrame(e11_metrics)
e11_subgroups=[]
for e11_name,e11_pred in e11_predictions.items():
    for e11_group,e11_mask in [('all_validation',np.ones(len(e11_val),dtype=bool)),
        ('training_unseen_term',~e11_val.term_seen_in_train.to_numpy(dtype=bool)),
        ('hate_heterogeneous_train_term',e11_val.hate_heterogeneous_in_train.to_numpy(dtype=bool)),
        ('three_annotators_only',e11_val.three_annotators.to_numpy(dtype=bool))]:
        e11_true=e11_yval[e11_mask];e11_out=e11_pred[e11_mask];e11_estimable=set(e11_true)=={0,1}
        e11_subgroups.append({'model':e11_name,'subgroup':e11_group,'examples':len(e11_true),
            'label_0':int(np.sum(e11_true==0)),'label_1':int(np.sum(e11_true==1)),
            'macro_f1':float(f1_score(e11_true,e11_out,labels=[0,1],average='macro',zero_division=0)) if e11_estimable else None,
            'macro_f1_estimable':e11_estimable})
e11_subgroups=pd.DataFrame(e11_subgroups)
print('Step 11: validation baselines complete')
print(e11_metrics[['model','examples','macro_f1','f1_label_0','f1_label_1','balanced_accuracy']].to_string(index=False))
print('TF-IDF features:',len(e11_vectorizer.vocabulary_))
print('These are results on the provisional validation split; the test partition was not used.')


## 3. Save Baseline Results and Run Settings

Run the next cell. Save validation predictions, overall and subgroup metrics, confusion counts, package versions and input hashes. Prediction tables contain example IDs and labels rather than sentences or glosses. Final thesis comparisons require common test IDs across models after protocol confirmation and the planned statistical tests.


In [ ]:
if 'e11_metrics' not in globals():raise RuntimeError('Run the first two Step 11 cells first.')
for e11_rel,e11_digest in e11_inputs.items():
    if hashlib.sha256((e11_root/e11_rel).read_bytes()).hexdigest()!=e11_digest:raise ValueError('Inputs changed. Saving stopped.')
e11_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
e11_out=e11_root/'results'/f'validation_baselines_11_{e11_stamp}'
e11_out.mkdir(parents=True,exist_ok=False)
e11_tables={'validation_predictions.csv':e11_prediction_table,
 'validation_metrics.csv':e11_metrics,'validation_subgroups.csv':e11_subgroups}
for e11_name,e11_table in e11_tables.items():e11_table.to_csv(e11_out/e11_name,index=False,encoding='utf-8-sig')
e11_config={'status':'exploratory_validation_baseline_on_provisional_split','input_sha256':e11_inputs,
 'split_draft':str(e10_out.relative_to(e11_root)),'training_ids':len(e11_train),'validation_ids':len(e11_val),
 'test_ids_evaluated':0,'train_majority_label':e11_majority,
 'term_prior':'NFKC+casefold+whitespace-normalized target term and POS; training-only vote majority, tie/unseen fallback to training global majority',
 'tfidf':{'analyzer':'word','ngram_range':[1,2],'min_df':2,'max_features':50000,'sublinear_tf':True,
          'lowercase':True,'strip_accents':'unicode','token_pattern':r'(?u)\b\w+\b','fit_on':'training_text_only'},
 'logreg':{'C':1.0,'class_weight':'balanced','max_iter':1000,'solver':'lbfgs','random_state':20261021},
 'model_text_exception':'U+01C3 replaced with same-length ! only for flagged model input; source sentence unchanged',
 'metrics':'per-class F1, macro F1 and balanced accuracy on validation IDs; zero_division=0',
 'versions':{'scikit-learn':e11_sklearn_version,'pandas':version('pandas'),'numpy':version('numpy')},
 'model_comparison_status':'BERT/HateBERT not yet trained','final_test_status':'sealed pending protocol confirmation'}
(e11_out/'baseline_protocol.json').write_text(json.dumps(e11_config,ensure_ascii=False,indent=2),encoding='utf-8')
e11_readme='\n'.join(['# Step 11 validation baselines',
 f'Training: {len(e11_train)}; validation: {len(e11_val)}. The test partition was not used for fitting, tuning or evaluation.',
 e11_metrics[['model','examples','macro_f1','f1_label_0','f1_label_1','balanced_accuracy']].to_string(index=False),
 'These metrics use the provisional Step 10 split and are provided for supervisor review.',
 'All three baselines use the same validation IDs. TF-IDF is fitted on training text only.',
 'Next: confirm preprocessing and splits, then train matched BERT and HateBERT target models.'])
(e11_out/'BASELINE_README.md').write_text(e11_readme,encoding='utf-8')
e11_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(e11_out.iterdir()) if p.is_file()}
(e11_out/'manifest.json').write_text(json.dumps(e11_manifest,indent=2),encoding='utf-8')
for e11_name,e11_digest in e11_manifest.items():
    if hashlib.sha256((e11_out/e11_name).read_bytes()).hexdigest()!=e11_digest:raise ValueError('Output verification failed.')
e11_check=pd.read_csv(e11_out/'validation_predictions.csv',dtype={'example_id':str})
if set(e11_check.example_id)!=set(e11_val.example_id) or len(e11_check)!=len(e11_val):raise ValueError('Prediction IDs differ from the validation IDs.')
print('Step 11: saved results verified')
print('Saved validation predictions, overall and subgroup metrics, settings and file hashes.')
print('Output folder:',str(e11_out.relative_to(e11_root)))


# HateWiC Step 12: Target Pooling and Geometry Smoke Tests

This is a **method implementation check** for the 24 October foundation checkpoint. Synthetic sentences and vectors test character spans, target subword positions, mean pooling and the three planned RO1 geometry measures. No BERT/HateBERT weights are loaded and no real-data representation results are produced.

## 1. Check Target-Subword Mean Pooling

Run the next cell. Use both official tokenizers loaded in Step 8. Synthetic cases cover repeated words, multiword expressions and case.


In [ ]:
if not all(k in globals() for k in ['e08_tokenizers','e08_encode','e08_target_alignment','e08_window','e08_check_prepared','e08_tokenizer_meta']):
    raise RuntimeError('Restore the tokenizers and alignment functions from Step 8 Sections 1-2. Do not repeat manual reviews.')
import numpy as np
from importlib.metadata import version

def e12_pool_target(hidden,positions):
    hidden=np.asarray(hidden,dtype=np.float64)
    positions=list(positions)
    if hidden.ndim!=2 or not positions or len(set(positions))!=len(positions):
        raise ValueError('Invalid hidden states or target positions.')
    if min(positions)<0 or max(positions)>=hidden.shape[0] or not np.isfinite(hidden).all():
        raise ValueError('Target positions are out of bounds or hidden states contain non-finite values.')
    return hidden[positions].mean(axis=0)

e12_cases=[('A cat meets another cat.','cat',1),
           ('The welfare queens arrived.','welfare queens',0),
           ('Geezers spoke.','Geezers',0)]
e12_pool_rows=[]
for e12_name,e12_tok in e08_tokenizers.items():
    for e12_text,e12_target,e12_occurrence in e12_cases:
        e12_starts=[i for i in range(len(e12_text)) if e12_text.startswith(e12_target,i)]
        e12_start=e12_starts[e12_occurrence];e12_end=e12_start+len(e12_target)
        e12_encoded=e08_encode(e12_tok,e12_text)
        e12_ix,e12_complete,e12_spill=e08_target_alignment(
            e12_text,e12_encoded['offset_mapping'],e12_start,e12_end)
        if not e12_complete or e12_spill:raise ValueError('Synthetic sentence alignment failed.')
        e12_win=e08_window(len(e12_encoded['input_ids']),e12_ix,126)
        if e12_win is None:raise ValueError('The synthetic target cannot fit within a 128-token window.')
        e12_positions,e12_length=e08_check_prepared(e12_tok,e12_encoded['input_ids'],e12_ix,
            e12_win[0],e12_win[1],128)
        e12_hidden=np.arange(e12_length*4,dtype=float).reshape(e12_length,4)
        e12_pooled=e12_pool_target(e12_hidden,e12_positions)
        if not np.allclose(e12_pooled,np.mean(e12_hidden[e12_positions],axis=0)):
            raise ValueError('Target mean pooling differs from the manual calculation.')
        if 0 in e12_positions or e12_length-1 in e12_positions:
            raise ValueError('Target pooling includes CLS or SEP.')
        e12_pool_rows.append({'backbone':e12_name,'case':e12_target,
            'target_subwords':len(e12_positions),'pooling_matches_manual':True})
if len(e12_pool_rows)!=2*len(e12_cases):raise ValueError('Synthetic checks are incomplete.')
print('Step 12: target-pooling checks passed;',len(e12_pool_rows),'checks.')
print('These checks use synthetic hidden states; model weights have not been loaded.')


## 2. Check Geometry Metrics

Run the next cell. L2-normalize vectors first. Define cosine separation as mean same-class pair similarity minus mean different-class pair similarity. Silhouette uses cosine distance; Davies–Bouldin uses Euclidean distance on normalized vectors. These synthetic checks verify calculations and metric direction, not real model performance.


In [ ]:
if 'e12_pool_rows' not in globals():raise RuntimeError('Run Step 12 Section 1 first.')
from sklearn.metrics import silhouette_score,davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

def e12_geometry(vectors,labels):
    vectors=np.asarray(vectors,dtype=np.float64)
    labels=np.asarray(labels,dtype=int)
    if vectors.ndim!=2 or len(vectors)!=len(labels) or not np.isfinite(vectors).all():
        raise ValueError('Invalid vectors or labels.')
    if len(np.unique(labels))!=2 or min(np.bincount(labels))<2:
        raise ValueError('Each of the two classes requires at least two examples.')
    norms=np.linalg.norm(vectors,axis=1)
    if np.any(norms<=0):raise ValueError('Zero vectors cannot be normalized.')
    vectors=vectors/norms[:,None]
    sims=cosine_similarity(vectors)
    same=[];different=[]
    for i in range(len(labels)):
        for j in range(i+1,len(labels)):
            (same if labels[i]==labels[j] else different).append(float(sims[i,j]))
    e12_within_between_gap=float(np.mean(same)-np.mean(different))
    return {'within_cosine':float(np.mean(same)),'between_cosine':float(np.mean(different)),
            'cosine_separation':e12_within_between_gap,
            'silhouette_cosine':float(silhouette_score(vectors,labels,metric='cosine')),
            'davies_bouldin_l2_normalized_euclidean':float(davies_bouldin_score(vectors,labels)),
            'within_pairs':len(same),'between_pairs':len(different)}

e12_vectors=np.array([[1.,0.],[.9,.1],[-1.,0.],[-.9,-.1]])
e12_labels=np.array([0,0,1,1])
e12_metric_check=e12_geometry(e12_vectors,e12_labels)
if not(e12_metric_check['cosine_separation']>0 and e12_metric_check['silhouette_cosine']>0
       and e12_metric_check['davies_bouldin_l2_normalized_euclidean']>=0):
    raise ValueError('Synthetic cluster metrics have an unexpected direction.')
if e12_metric_check['within_pairs']!=2 or e12_metric_check['between_pairs']!=4:
    raise ValueError('Incorrect number of example pairs.')
print('Step 12: geometry metric checks passed.')
print('Synthetic-vector results (code verification only):',e12_metric_check)


## 3. Save Implementation Checks

Run the next cell. Save synthetic test results, metric definitions, tokenizer versions and file hashes, without source data or real-model performance claims.


In [ ]:
if 'e12_metric_check' not in globals():raise RuntimeError('Run the first two Step 12 cells first.')
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,platform
e12_root=Path('/content/drive/MyDrive/HateWiC_Project')
e12_out=e12_root/'results'/('foundation_smoke_12_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
e12_out.mkdir(parents=True,exist_ok=False)
e12_report={'status':'synthetic_method_smoke_test_only','model_weights_loaded':False,
    'real_data_geometry_evaluated':False,'synthetic_cases':len(e12_cases),
    'tokenizer_checks':e12_pool_rows,'synthetic_metric_check':e12_metric_check,
    'cosine_separation_definition':'mean same-class pair cosine minus mean different-class pair cosine; unordered distinct pairs',
    'silhouette_definition':'cosine distance on L2-normalized vectors',
    'davies_bouldin_definition':'Euclidean distance on L2-normalized vectors',
    'pooling_definition':'arithmetic mean of all target subword hidden states; excludes CLS/SEP',
    'python':platform.python_version(),'numpy':np.__version__,
    'scikit_learn_live':__import__('sklearn').__version__,
    'tokenizers':e08_tokenizer_meta}
(e12_out/'foundation_smoke_protocol.json').write_text(json.dumps(e12_report,ensure_ascii=False,indent=2),encoding='utf-8')
e12_manifest={'foundation_smoke_protocol.json':hashlib.sha256((e12_out/'foundation_smoke_protocol.json').read_bytes()).hexdigest()}
(e12_out/'manifest.json').write_text(json.dumps(e12_manifest,indent=2),encoding='utf-8')
if hashlib.sha256((e12_out/'foundation_smoke_protocol.json').read_bytes()).hexdigest()!=e12_manifest['foundation_smoke_protocol.json']:
    raise ValueError('Step 12 output hash verification failed.')
print('Step 12: saved results verified')
print('Output folder:',str(e12_out.relative_to(e12_root)))
print('Next: prepare the 24 October supervisor update and confirm the outstanding protocol decisions.')
